# E18 — S1-60K on vLLM 0.32.0, 20 streams

**User (2026-10-27):** S1-60K (LB 34.58, sha `ed530071`) unchanged except the engine: vLLM 0.32.0 instead of 0.19.0, and
20 streams instead of 16. Poetiq, prompts, 60,000 + 8,192 tokens, sampling, queue, KV gate and deadlines are S1's.

- The solver, report, Poetiq core, task order, install (CELL 2) and CELL 0 cells are **byte-identical** to S1-60K.
- CELL 5 runs vLLM 0.32.0 as an OpenAI server from a venv (`foysalemonshanto/vllm-0-32-0-cu13-wheelhouse`, the
  Experiment 14 VMS launch) with S1's engine flags, and gives the solver the same `engine.add_request / step /
  abort_request`, `llm.get_metrics()` and `SamplingParams` names through a shim. Prompts are S1's token ids.
- Ladder: S1 flags + language-model-only → the exact VMS `fp8_mtp3_s20` flags → VMS with Triton attention → S1's own
  in-process vLLM 0.19 engine cell (verbatim).
- Commit run: `E18SMOKE` = 20 eval tasks, 45 min (engine, 20 concurrent units, drain, verification, tok/s). The hidden
  rerun is `RERUN_ALL` (12 h), unchanged.

---

# Q27 E10 — 86K × Poetiq Train-Verified Program Solver

**Base:** User-supplied Q27-XHIGH-v1_4_4_E10_BASE_86K.ipynb (SHA256 `530cb119...`).

**Frozen architecture:** 86K Q27 local vLLM → Poetiq task-level program synthesis → isolated Python execution on visible train pairs → exact/soft feedback → iterative refinement → expert voting/diversity → two ranked attempt grids.

**User decisions:** 77,824 thinking + 8,192 answer; original task order (no Codex ordering); no E9 tail resample; original 4×L4 FP8-KV/MTP3 engine initialization; do not insert old Q27 grid ranker. Poetiq donor default: one expert, 10 iterations, 5 feedback examples, temperature 1.0. Config supports 2/8+ experts and prompt variants 1/2/3 without editing code.

**Source:** [poetiq-ai/poetiq-arc-agi-solver](https://github.com/poetiq-ai/poetiq-arc-agi-solver), MIT. This is a local-vLLM adaptation, not a byte-for-byte Poetiq clone or a measured Poetiq accuracy claim. Kaggle has no internet; all runtime code is embedded here.

**STATUS:** CPU-validated prototype; requires 4×L4 GPU validation. No GPU or Kaggle submission run by the authoring assistant.


**R4T (2026-10-14):** R4 + the full thinking of every iteration saved to `q27x/poetiq_thinking/` (logging only).


**R5 + EVAL120 (2026-10-14):** commit run = all 120 evaluation tasks / 172 test inputs on the hidden-run clock (admit until 11:34, drain 11:36, hard stop 11:44). Everything else is R5 unchanged (77,824 + 8,192, full thinking saved to q27x/poetiq_thinking/). `Q27X_PROFILE=EVAL12` gives the R5 default.


**S1-60K (2026-10-16), submission build and the Experiment 11 base:** R5 + ORDER2 queue + F1/F3/A2/F2/C1 fixes + KV admission gate (0.95, >= 12 running) + weight prefetch + 12 h clock from kernel start (last admission 11:45, drain 11:47, hard stop 11:55; R5 left 16 min unused). **Thinking 60,000 + 8,192** (user: LB test of the 60k threshold against R5's 77,824; `Q27X_THINK_BUDGET` overrides). Temperature 1.0 unchanged. Commit run = 4 quick eval tasks (`S1COMMIT4`, about 1 h) so the version can be submitted; the hidden rerun solves every task. `Q27X_PROFILE=EVAL120` gives the full evaluation run.


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "E18SMOKE").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # E18 (2026-10-27): the commit run of the vLLM 0.32 / 20-stream build: 20 eval tasks (20 concurrent units at 1 expert),
    # 45 min; engine ready ~10-12 min, admission until 35 min, drain 36 min, hard stop 42 min
    "E18SMOKE":  {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a", "0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3"],
                  "time_budget_sec": 45 * 60, "admit_frac": 0.80, "drain_frac": 0.90, "hard_margin_sec": 3 * 60,
                  "verify_margin_sec": 1 * 60, "drain_margin_sec": 6 * 60, "admit_margin_sec": 1 * 60},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))
# E18: the vLLM 0.32.0 venv wheelhouse (the engine); WHEELHOUSE above keeps the tokenizer stack + the vLLM 0.19 fallback
E18_VLLM_WHEELHOUSE = Path(os.environ.get("E18_VLLM_WHEELHOUSE", "/kaggle/input/datasets/foysalemonshanto/vllm-0-32-0-cu13-wheelhouse"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # S1-60K: LB test of 60,000 (R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 20))   # E18: 16 -> 20 (VMS fp8_mtp3_s20 PASS)   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
CONFIG["e18"] = {"engine": "vLLM 0.32.0 server (venv) + in-process shim; S1's vLLM 0.19 engine as the last fallback",
                 "vllm_wheelhouse": str(E18_VLLM_WHEELHOUSE), "max_num_seqs": MAX_NUM_SEQS, "base": "S1-60K ed530071"}
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','5'))
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','1')=='1'
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
if os.environ.get("Q27X_SKIP_INSTALL") != "1":
    try:
        import shlex as _shlex
        _wfiles = sorted(str(p) for p in MODEL_PATH.glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install([f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                             if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]])
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
# E10 Poetiq donor adaptation: task formatting, code extraction, sandbox, feedback, voting.
# Donor: https://github.com/poetiq-ai/poetiq-arc-agi-solver (MIT)
# Donor originals: arc_agi/solve_coding.py, solve_parallel_coding.py, sandbox.py, prompts.py
import numpy as np
import json, re, sys, subprocess, tempfile, textwrap, math, time, os, string
from pathlib import Path

# Donor prompt variants adapted solely to the local Q27 chat interface.
# The task wording, Python transform contract, and refinement feedback follow Poetiq.
POETIQ_SOLVER_PROMPTS = {
  1: '''You are an expert in solving Abstract Reasoning Corpus (ARC) tasks by writing Python code. Your goal is to analyze input-output examples and create a 'transform' function that correctly transforms any given input grid into the corresponding output grid.
Analyze the examples: objects, colors, spatial relations, shapes, lines, symmetries, sizes. Formulate one consistent rule, prioritize simple general rules. Implement a Python function `transform(grid: np.ndarray) -> np.ndarray` with NumPy or standard libraries. Return a single complete markdown ```python code block containing the function; include needed imports. No __main__ block. The program must work on all training inputs and all test inputs. Do not hardcode specific test outputs.
PROBLEM:\n$$problem$$''',
  2: '''You are a world-class expert in solving ARC tasks and an expert Python coder. Analyze every example carefully, then hypothesize a SINGLE consistent rule applicable to all examples. Consider colors, connected components, rotations, reflections, object motion, addition/removal, spatial patterns and symmetries. Test and refine the hypothesis to match all example outputs. Implement `def transform(grid: np.ndarray) -> np.ndarray` using NumPy (cv2 and standard libraries may be used if available). Your answer must contain a single valid markdown ```python code block defining the function and any imports. Explain briefly before the code. Do not output a __main__ block.
PROBLEM:\n$$problem$$''',
  3: '''You are a world-class ARC reasoning researcher and master Python coder. Inspect the dimensions, positions, colors, objects, neighborhoods and geometry in all demonstration input/output pairs. Generate possible hypotheses; use the simplest rule explaining all pairs. Translate it into executable, documented Python code using NumPy and optionally cv2/standard libraries. Handle edge cases, unexpected shapes and colors. Return one markdown ```python code block containing imports and `def transform(grid: np.ndarray) -> np.ndarray`, plus a short explanation. Do not include a __main__ block. The same rule must generalize to each Challenge input.
PROBLEM:\n$$problem$$'''
}
POETIQ_FEEDBACK_PROMPT = '''**EXISTING PARTIAL/INCORRECT SOLUTIONS:**\nFollowing are some of the best, though not completely correct, solutions so far. For each solution, code, training-example feedback, and a score from 0 to 1 are provided. Study the code and the feedback and produce a NEW complete Python `transform` solution fixing all the issues. Follow the exact output format.\n$$feedback$$'''

def poetiq_diagram(g):
    return '\n'.join(' '.join(str(int(v)) for v in row) for row in g)

def poetiq_format_problem(task, shuffle=False, seed=0):
    train = list(task['train'])
    if shuffle and len(train) > 1:
        perm = np.random.default_rng(seed).permutation(len(train))
        train = [train[int(i)] for i in perm]
    chunks=[]
    for i, item in enumerate(train,1):
        chunks.append(f'Example #{i}\nInput:\n<Diagram>\n{poetiq_diagram(item["input"])}\n</Diagram>\nOutput:\n<Diagram>\n{poetiq_diagram(item["output"])}\n</Diagram>')
    for i,item in enumerate(task['test'],1):
        chunks.append(f'Challenge #{i}\nInput:\n<Diagram>\n{poetiq_diagram(item["input"])}\n</Diagram>')
    return '\n\n'.join(chunks)

def poetiq_parse_code(response):
    # S1 (F1): the LAST ```python block that parses and defines transform(); R5 took the first block, so a short
    # illustrative snippet written before the real solution was the one executed. Falls back to the R5 choice.
    import ast as _ast
    blocks=re.findall(r'```python\s*(.*?)```',response or '',re.DOTALL|re.IGNORECASE)
    for b in reversed(blocks):
        b=b.strip()
        try:
            tree=_ast.parse(b)
        except (SyntaxError,ValueError):
            continue
        if any(isinstance(n,_ast.FunctionDef) and n.name=='transform' for n in tree.body):
            return b
    return blocks[0].strip() if blocks else None

def poetiq_create_examples(solutions, max_examples=5, improving_order=True):
    if not solutions: return ''
    chosen=sorted(solutions,key=lambda z:z['score'], reverse=True)[:max_examples]
    if improving_order: chosen.reverse()
    return '\n\n'.join(f'<solution_{k}>\n<solution_code>\n```python\n{s["code"]}\n```\n</solution_code>\n<solution_evaluation>\n{s["feedback"]}\n</solution_evaluation>\n<solution_score>\n{s["score"]:.2f}\n</solution_score>\n</solution_{k}>' for k,s in enumerate(chosen,1))

def poetiq_build_message(task,iteration,expert_idx,solutions,variant=1,selection_probability=1.,max_solutions=5,improving_order=True,shuffle=True,seed=0):
    # Poetiq's seeded example shuffling and best-history feedback.
    problem=poetiq_format_problem(task, shuffle, seed+iteration)
    msg=POETIQ_SOLVER_PROMPTS[int(variant)].replace('$$problem$$',problem)
    if solutions and max_solutions > 0:
        rng=np.random.default_rng(seed+100000*expert_idx+iteration)
        subset=[s for s in solutions if rng.random()<selection_probability]
        if subset:
            msg+='\n\n'+POETIQ_FEEDBACK_PROMPT.replace('$$feedback$$',poetiq_create_examples(subset,max_solutions,improving_order))
    return msg

def poetiq_normalize_grid(g):
    """Lossless ARC palette normalization, rejecting booleans and fractions."""
    if not isinstance(g, list) or not 1 <= len(g) <= 30:
        return None
    if not all(isinstance(row,list) and 1 <= len(row) <= 30 for row in g):
        return None
    if len({len(row) for row in g}) != 1:
        return None
    normalized=[]
    for row in g:
        rr=[]
        for v in row:
            if isinstance(v, (bool, np.bool_)):
                return None
            if isinstance(v,(int,np.integer)):
                value=int(v)
            elif isinstance(v,(float,np.floating)) and math.isfinite(float(v)) and float(v).is_integer():
                value=int(v)
            else:
                return None
            if not 0 <= value <= 9:
                return None
            rr.append(value)
        normalized.append(rr)
    return normalized


def poetiq_valid_grid(g):
    return poetiq_normalize_grid(g) is not None


def poetiq_grid_from_output(text):
    try:
        return poetiq_normalize_grid(json.loads(text))
    except (TypeError,ValueError,OverflowError):
        return None


# Poetiq subprocess execution adapted for Kaggle Docker without unshare.
# UID separation + parent limits + no-new-privs + seccomp are enforced.
# These do not create a mount namespace. World-readable host files are NOT
# guaranteed inaccessible; see the README's explicit residual risk.
_POETIQ_TRUSTED_RUNNER = '"""Trusted Poetiq runner for Kaggle\'s namespace-restricted Docker containers.\n\nNot a mount namespace. It provides UID separation and syscall/resource limits;\nworld-readable host paths are NOT guaranteed unreadable without a mount or LSM.\n"""\nimport sys, os, json, ctypes, errno, resource, traceback\n# Import numerical/scientific libraries before syscall filtering.\nimport numpy as np\ntry:\n    import scipy\nexcept ImportError:\n    scipy = None\ntry:\n    import cv2\nexcept ImportError:\n    cv2 = None\n\n_RESULT_MARKER = \'__E10_PROGRAM_RESULT__\'\n\n\ndef drop_privileges_and_limits():\n    # Trusted code runs before compiling or executing model-generated Python.\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Root is required for irreversible UID/GID drop\')\n    os.setgroups([])\n    os.setgid(65534)\n    os.setuid(65534)\n    cpu = max(2,min(120,int(os.environ.get(\'E10_CHILD_CPU\',\'10\'))))\n    mem = 4096*1024*1024\n    resource.setrlimit(resource.RLIMIT_CPU,(cpu,cpu+1))\n    resource.setrlimit(resource.RLIMIT_AS,(mem,mem))\n    resource.setrlimit(resource.RLIMIT_FSIZE,(1024*1024,1024*1024))\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    resource.setrlimit(resource.RLIMIT_NOFILE,(64,64))\n    resource.setrlimit(resource.RLIMIT_NPROC,(64,64))\n\n\ndef harden_process():\n    libc = ctypes.CDLL(None, use_errno=True)\n    if libc.prctl(38, 1, 0, 0, 0) != 0:\n        raise RuntimeError(\'PR_SET_NO_NEW_PRIVS failed\')\n    lib = ctypes.CDLL(\'libseccomp.so.2\', use_errno=True)\n    lib.seccomp_init.argtypes = [ctypes.c_uint32]\n    lib.seccomp_init.restype = ctypes.c_void_p\n    lib.seccomp_syscall_resolve_name.argtypes = [ctypes.c_char_p]\n    lib.seccomp_syscall_resolve_name.restype = ctypes.c_int\n    lib.seccomp_rule_add.argtypes = [ctypes.c_void_p, ctypes.c_uint32, ctypes.c_int, ctypes.c_uint]\n    lib.seccomp_rule_add.restype = ctypes.c_int\n    lib.seccomp_load.argtypes = [ctypes.c_void_p]\n    lib.seccomp_load.restype = ctypes.c_int\n    lib.seccomp_release.argtypes = [ctypes.c_void_p]\n    lib.seccomp_release.restype = None\n    ctx = lib.seccomp_init(0x7fff0000)\n    if not ctx:\n        raise RuntimeError(\'seccomp_init failed\')\n    try:\n        # Blocking exec/spawn/network/ptrace/mount remains compatible with\n        # numpy/scipy/cv2. Do not block clone(thread) indiscriminately.\n        blocked = (\n            \'execve\',\'execveat\',\'fork\',\'vfork\',\n            \'socket\',\'socketpair\',\'connect\',\'bind\',\'listen\',\'accept\',\'accept4\',\n            \'mount\',\'umount2\',\'pivot_root\',\'chroot\',\'unshare\',\'setns\',\n            \'ptrace\',\'process_vm_readv\',\'process_vm_writev\',\'open_by_handle_at\',\n            \'reboot\',\'bpf\',\'keyctl\',\'setuid\',\'setgid\',\'setreuid\',\'setregid\',\n            \'setresuid\',\'setresgid\',\n        )\n        for name in blocked:\n            nr = lib.seccomp_syscall_resolve_name(name.encode())\n            if nr >= 0:\n                if lib.seccomp_rule_add(ctx, 0x00050000 | errno.EPERM, nr, 0):\n                    raise RuntimeError(\'seccomp rule failed: \' + name)\n        # glibc falls back from clone3 to clone only on ENOSYS, not EPERM.\n        # Keep process creation blocked by the existing clone/CLONE_THREAD rule.\n        clone3_nr = lib.seccomp_syscall_resolve_name(b\'clone3\')\n        if clone3_nr >= 0:\n            rc = lib.seccomp_rule_add(ctx, 0x00050000 | errno.ENOSYS, clone3_nr, 0)\n            if rc:\n                raise RuntimeError(\'seccomp clone3 restriction failed: \' + str(rc))\n        # clone() flags: require CLONE_THREAD when creating a thread.\n        # This blocks ordinary child processes without disabling numerical threads.\n        clone_nr = lib.seccomp_syscall_resolve_name(b\'clone\')\n        if clone_nr >= 0:\n            class ArgCmp(ctypes.Structure):\n                _fields_=[(\'arg\',ctypes.c_uint),(\'op\',ctypes.c_uint),\n                          (\'datum_a\',ctypes.c_uint64),(\'datum_b\',ctypes.c_uint64)]\n            # seccomp_rule_add_array(ctx, action, syscall, n, &cmp)\n            lib.seccomp_rule_add_array.argtypes=[ctypes.c_void_p,ctypes.c_uint32,ctypes.c_int,\n                                                   ctypes.c_uint,ctypes.POINTER(ArgCmp)]\n            lib.seccomp_rule_add_array.restype=ctypes.c_int\n            # SCMP_CMP_MASKED_EQ=7; deny clone if CLONE_THREAD bit is zero.\n            cmp=ArgCmp(0,7,0x10000,0)\n            rc=lib.seccomp_rule_add_array(ctx,0x00050000|errno.EPERM,clone_nr,1,ctypes.byref(cmp))\n            if rc:\n                raise RuntimeError(\'seccomp clone restriction failed: \'+str(rc))\n        rc=lib.seccomp_load(ctx)\n        if rc:\n            raise RuntimeError(\'seccomp_load failed \'+str(rc))\n    finally:\n        lib.seccomp_release(ctx)\n\n\ndef main():\n    raw = json.load(sys.stdin)\n    program = raw[\'code\']\n    grid = raw[\'input\']\n    if not isinstance(program, str) or len(program)>250000:\n        raise ValueError(\'invalid code length\')\n    drop_privileges_and_limits()\n    harden_process()\n    namespace = {\'__name__\': \'poetiq_candidate\', \'__builtins__\': __builtins__, \'np\': np}\n    try:\n        exec(compile(program, \'<poetiq_candidate>\', \'exec\'), namespace)\n        result=namespace[\'transform\'](np.array(grid, dtype=int))\n        if hasattr(result,\'tolist\'):\n            result=result.tolist()\n        payload={\'ok\':True,\'result\':result}\n    except BaseException as ex:\n        payload={\'ok\':False,\'error\':(type(ex).__name__+\': \'+str(ex))[:1200]}\n    try:\n        print(_RESULT_MARKER + json.dumps(payload,separators=(\',\',\':\')),flush=True)\n    except BaseException as ex:\n        print(_RESULT_MARKER + json.dumps({\'ok\':False,\'error\':\'JSON output error: \'+str(ex)[:500]}),flush=True)\n\nif __name__==\'__main__\':\n    try:\n        main()\n    except BaseException as ex:\n        sys.stderr.write(\'E10_RUNNER_SETUP_ERROR: \'+repr(ex)+\'\\n\')\n        sys.exit(110)\n'



def poetiq_sandbox_run(code, grid, timeout_s=5.0):
    import signal
    timeout_s=max(.1,float(timeout_s))
    if os.geteuid()!=0:
        return False,'sandbox unavailable: cannot drop root privileges to UID 65534'
    try:
        import ctypes
        ctypes.CDLL('libseccomp.so.2')
    except Exception as e:
        return False,'sandbox unavailable: libseccomp: '+str(e)[:200]
    with tempfile.TemporaryDirectory(prefix='e10_poetiq_r4_') as td:
        # Child cwd and TMPDIR are an isolated per-program scratch folder.
        # It is NOT a mount-based read sandbox; UID/seccomp prevent spawning,
        # outbound sockets and writing to root-owned Kaggle checkpoints.
        os.chown(td,65534,65534)
        os.chmod(td,0o700)
        output=Path(td)/'stdout.log'; errors=Path(td)/'stderr.log'
        env={'PYTHONHASHSEED':'0','PYTHONDONTWRITEBYTECODE':'1',
             'CUDA_VISIBLE_DEVICES':'','OMP_NUM_THREADS':'1',
             'OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1',
             'NUMEXPR_NUM_THREADS':'1','TMPDIR':td,
             'HOME':td, 'PATH':'/usr/bin:/bin'}
        env['E10_CHILD_CPU']=str(max(2,min(120,math.ceil(timeout_s)+1)))
        cmd=[sys.executable,'-I','-B','-c',_POETIQ_TRUSTED_RUNNER]
        try:
            with output.open('wb') as outf,errors.open('wb') as errf:
                proc=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=outf,stderr=errf,
                      cwd=td,env=env,start_new_session=True,close_fds=True)
                try:
                    proc.communicate(input=json.dumps({'code':code,'input':grid}).encode(), timeout=timeout_s)
                except subprocess.TimeoutExpired:
                    try:os.killpg(proc.pid,signal.SIGKILL)
                    except ProcessLookupError:pass
                    proc.communicate(timeout=2)
                    return False,'sandbox timeout'
            if output.stat().st_size>1024*1024 or errors.stat().st_size>1024*1024:
                return False,'sandbox output limit exceeded'
            sout=output.read_text(encoding='utf-8',errors='replace')
            serr=errors.read_text(encoding='utf-8',errors='replace')
            if proc.returncode:
                return False,('sandbox process failure '+str(proc.returncode)+': '+(serr or sout))[-1500:]
            marker='__E10_PROGRAM_RESULT__'
            pos=sout.rfind(marker)
            if pos<0:
                return False,'sandbox missing result: '+sout[-800:]
            try: payload=json.loads(sout[pos+len(marker):].strip())
            except ValueError:
                return False,'sandbox malformed result: '+sout[-800:]
            if not payload.get('ok'):
                return False,str(payload.get('error','program error'))[:1200]
            return True,json.dumps(payload.get('result'))
        except Exception as exc:
            return False,('sandbox startup failure '+type(exc).__name__+': '+str(exc))[:1000]

def poetiq_array_diff(pred, truth):
    return '\n'.join(' '.join(str(int(a)) if a==b else f'{int(a)}/{int(b)}' for a,b in zip(pr,tr))
                     for pr,tr in zip(pred,truth))

def poetiq_verify_code(code,task,timeout_s=5.0):
    train_results=[]; per=[]
    for i,ex in enumerate(task['train']):
        ok,payload=poetiq_sandbox_run(code,ex['input'],timeout_s)
        # As in Poetiq, evaluate *partial training correctness* even when
        # predicted integer values are outside ARC 0..9; validate test grids separately.
        grid=None
        if ok:
            try:
                raw=json.loads(payload)
                # Never coerce floats, strings or bools into integer grids:
                # that can falsely mark a program train-verified and stop
                # Poetiq refinement while test candidates are invalid.
                grid=poetiq_normalize_grid(raw)
            except (TypeError,ValueError,IndexError,OverflowError):pass
        truth=ex['output']; exact=grid==truth if grid is not None else False
        if exact: soft=1.0; feedback=f'Solves Example #{i+1} correctly.'
        elif grid is None: soft=0.0;feedback=f'Solves Example #{i+1} incorrectly. Invalid or unavailable grid. Error: {payload[-600:]}'
        elif len(grid)!=len(truth) or len(grid[0])!=len(truth[0]):
            soft=0.0;feedback=f'Solves Example #{i+1} incorrectly. Shape mismatch: predicted {(len(grid),len(grid[0]))}, expected {(len(truth),len(truth[0]))}.'
        else:
            soft=sum(a==b for rowa,rowb in zip(grid,truth) for a,b in zip(rowa,rowb))/(len(truth)*len(truth[0]))
            feedback=f'Solves Example #{i+1} incorrectly. Wrong cells appear as prediction/correct:\n```\n{poetiq_array_diff(grid,truth)}\n```\nOutput accuracy: {soft:.2f}'
        per.append(soft);train_results.append({'success':bool(exact),'soft_score':soft,'feedback':feedback,'error':None if ok else payload})
    test_results=[]
    for ex in task['test']:
        ok,payload=poetiq_sandbox_run(code,ex['input'],timeout_s)
        grid=poetiq_grid_from_output(payload) if ok else None
        test_results.append({'grid':grid,'error':None if grid is not None else payload[-600:]})
    passed=all(r['success'] for r in train_results)
    test_usable=all(poetiq_valid_grid(r.get('grid')) for r in test_results)
    feedback='\n\n'.join(r['feedback'] for r in train_results)
    # Test-input truth is NEVER consulted: only runtime/format diagnostics.
    if not test_usable:
        problems=[f'Challenge #{i+1}: {r.get("error") or "invalid grid shape/palette"}'
                  for i,r in enumerate(test_results) if not poetiq_valid_grid(r.get('grid'))]
        feedback += '\n\nTEST EXECUTION / FORMAT ERRORS (no test answers are known):\n'+\
                    '\n'.join(problems)
    return {'train_results':train_results,'test_results':test_results,
            'passed':passed,'test_usable':test_usable,
            'verified_usable':bool(passed and test_usable),
            'score':float(sum(per)/len(per)) if per else 0.0,
            'feedback':feedback}

def poetiq_test_key(test_results):
    return json.dumps([r.get('grid') for r in test_results],separators=(',',':'))

def poetiq_mean_soft(res):
    trs=res.get('train_results',[])
    return sum(t.get('soft_score',0.0) for t in trs)/len(trs) if trs else 0.0

def poetiq_vote(expert_results,use_new_voting=True,count_failed_matches=True,iters_tiebreak=False,low_to_high_iters=False):
    # Poetiq's solve_parallel_coding grouping and diversity-first ranking semantics.
    passers={}; failures={}
    for result in expert_results:
        key=poetiq_test_key(result['test_results'])
        (passers if result['passed'] else failures).setdefault(key,[]).append(result)
    if use_new_voting and count_failed_matches:
        for key in list(failures):
            if key in passers:
                passers[key].extend(failures.pop(key))
    groups=list(passers.values())
    if iters_tiebreak:
        for group in groups:group.sort(key=lambda z:z.get('iteration',0), reverse=not low_to_high_iters)
        groups.sort(key=lambda g:g[0].get('iteration',0),reverse=low_to_high_iters)
    groups.sort(key=len,reverse=True)
    for group in failures.values():group.sort(key=poetiq_mean_soft,reverse=True)
    failed=list(failures.values());failed.sort(key=lambda g:(len(g),poetiq_mean_soft(g[0])),reverse=True)
    if use_new_voting:
        return ([g[0] for g in groups if g]+[g[0] for g in failed if g]+
                [m for g in groups for m in g[1:]]+[m for g in failed for m in g[1:]])
    return ([g[0] for g in groups if g]+sorted([r for g in failed for r in g],key=poetiq_mean_soft,reverse=True)
            +[m for g in groups for m in g[1:]])

def poetiq_two_attempts(task, ranked):
    # Poetiq-ranked program outputs supply both attempts; no old grid policy.
    out=[]
    for qi,test in enumerate(task['test']):
        grids=[]
        for r in ranked:
            t=r['test_results']
            g=t[qi]['grid'] if qi<len(t) else None
            if poetiq_valid_grid(g) and g not in grids:grids.append(g)
        # S1 (A2): never repeat attempt_1 (R5 did in 134/172 inputs); the next distinct grid, else the test input,
        # else its mirror
        mirror=[list(reversed(row)) for row in test['input']]
        if grids:
            a1=grids[0]
            a2=next((g for g in grids[1:]+[test['input'],mirror] if g!=a1),grids[0])
        else:
            a1=test['input'] # only if no valid program output; required schema
            a2=mirror if mirror!=a1 else test['input']
        out.append({'attempt_1':a1,'attempt_2':a2})
    return out

# --- DONOR ORIGINAL PROMPTS BELOW ---
# Source: poetiq-ai/poetiq-arc-agi-solver/arc_agi/prompts.py (MIT)
# Verbatim donor prompt texts, then loaded into Poetiq adaptation core.
SOLVER_PROMPT_1 = '''
You are an expert in solving Abstract Reasoning Corpus (ARC) tasks by writing Python code. Your goal is to analyze input-output examples and create a 'transform' function that correctly transforms any given input grid into the corresponding output grid.

Here's how to approach the problem:

**1. Analyze the Examples:**
  *   Identify the key objects in the input and output grids (e.g., shapes, lines, regions).
  *   Determine the relationships between these objects (e.g., spatial arrangement, color, size).
  *   Identify the operations that transform the input objects and relationships into the output objects and relationships (e.g., rotation, reflection, color change, object addition/removal).
  *   Consider the grid dimensions, symmetries, and other visual features.

**2. Formulate a Hypothesis:**
  *   Based on your analysis, formulate a transformation rule that works consistently across all examples.
  *   Express the rule as a sequence of image manipulation operations.
  *   Prioritize simpler rules first.
  *   Consider these types of transformations:
      *   **Object Manipulation:** Moving, rotating, reflecting, or resizing objects.
      *   **Color Changes:** Changing the color of specific objects or regions.
      *   **Spatial Arrangements:** Rearranging the objects in a specific pattern.
      *   **Object Addition/Removal:** Adding or removing objects based on certain criteria.

**3. Implement the Code:**
  *   Write a Python function called `transform(grid: np.ndarray) -> np.ndarray` that implements your transformation rule.
  *   Use NumPy for array manipulations. Other standard libraries are also available.
  *   Write modular code with clear variable names and comments to explain the logic behind each step.
  *   Document your code clearly, explaining the transformation rule in the docstring.
  *   Handle edge cases and invalid inputs gracefully.

**4. Test and Refine:**
  *   Test your code on all examples. If it fails for any example, refine your hypothesis and code.
  *   Use debugging techniques to identify and fix errors.
  *   Ensure your code handles edge cases and invalid inputs gracefully.

**5. Output:**
  *   Provide a brief explanation of your solution.
  *   Include the complete Python code for the `transform` function within a single markdown code block.
  *   Do not include any `__name__ == "__main__"` block or any code outside the function definition.

**Examples:**

**Example 1:**

**Input:**
```
[[1, 1, 1],
[1, 0, 1],
[1, 1, 1]]
```

**Output:**
```
[[0, 0, 0],
[0, 1, 0],
[0, 0, 0]]
```

**Explanation:**
Replace the border with 0s.

**Code:**
```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
  """Replace the border with 0s."""
  grid[0, :] = 0
  grid[-1, :] = 0
  grid[:, 0] = 0
  grid[:, -1] = 0
  return grid
```

**Example 2:**

**Input:**
```
[[1, 2, 3],
[4, 5, 6],
[7, 8, 9]]
```

**Output:**
```
[[9, 8, 7],
[6, 5, 4],
[3, 2, 1]]
```

**Explanation:**
Reverse the order of elements in each row and then reverse the order of the rows themselves.

**Code:**
```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
  """Reverses the order of elements in each row and then reverses the order of the rows."""
  new_grid = grid[:, ::-1][::-1]
  return new_grid
```

**Example 3:**

**Input:**
```
[[0, 0, 0, 0, 0],
[0, 1, 1, 1, 0],
[0, 1, 0, 1, 0],
[0, 1, 1, 1, 0],
[0, 0, 0, 0, 0]]
```

**Output:**
```
[[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 1, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]]
```

**Explanation:**
Keep only the center pixel if it is 1, otherwise make the grid all zeros.

**Code:**
```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
  """Keep only the center pixel if it is 1, otherwise make the grid all zeros."""
  center_row, center_col = grid.shape[0] // 2, grid.shape[1] // 2
  if grid[center_row, center_col] == 1:
      new_grid = np.zeros_like(grid)
      new_grid[center_row, center_col] = 1
      return new_grid
  else:
      return np.zeros_like(grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''
SOLVER_PROMPT_2 = '''
You are a world-class expert in solving Abstract Reasoning Corpus (ARC) tasks. Your approach is methodical, creative, and highly effective. You are also a master Python coder, producing elegant, efficient, and well-documented solutions.

Your goal is to analyze a set of input-output examples and devise a Python function that accurately transforms any input grid into its corresponding output grid. The key is to identify a *single, consistent transformation rule* that generalizes across *all* examples. Do not give up until you find a correct solution.

Follow this iterative process:

**Part 1: Initial Analysis and Hypothesis Generation**

1.  **Example Inspection:** Carefully examine the input and output grids for each example. Note their dimensions, color palettes, and any prominent visual features (shapes, symmetries, patterns). Use visualization techniques to aid your analysis.
2.  **Transformation Hypotheses:** Formulate several candidate transformation rules. Start with simpler rules and gradually increase complexity. Consider these categories:
    *   **Color Transformations:** Replacing colors based on specific criteria (e.g., adjacency, frequency). For example, replace all 0s with 1s, or replace the most frequent color with the least frequent color.
    *   **Object Isolation:** Identifying and isolating objects based on color, shape, or position. For example, extract the largest connected component of a certain color, or isolate objects based on their spatial relationships.
    *   **Spatial Operations:** Rotating, reflecting, resizing, or moving objects. For example, rotate the grid by 90 degrees, reflect the grid horizontally or vertically, or resize the grid by a certain factor.
    *   **Pattern Generation:** Replicating or extending existing patterns. For example, repeat a certain pattern across the grid, or generate a new pattern based on the existing patterns.
3.  **Symmetry Analysis:** Identify any symmetries (rotational, reflectional) in the input and output grids. Determine if the transformation preserves or alters these symmetries.

**Part 2: Iterative Testing and Refinement**

1.  **Code Implementation:** Implement your strongest candidate rule as a Python function. The function *must* accept a 2D numpy array as input and return a 2D numpy array as output.
2.  **Rigorous Testing:** Test your code against *all* training examples. A single failure indicates an incorrect rule.
3.  **Feedback Analysis:** If your code fails, carefully analyze the feedback. Identify the specific examples that failed and the nature of the errors. Use print statements to debug intermediate values and verify your assumptions.
4.  **Hypothesis Refinement:** Based on the feedback, refine your transformation rule. This may involve adjusting parameters, adding new conditions, or discarding the rule altogether and starting with a new hypothesis.
5.  **Repeat:** Continue this iterative process of coding, testing, and refining until you find a rule that works for all training examples. Do not give up until you find a correct solution.

**Part 3: Coding Guidelines**

1.  **Available Libraries:** You can use `numpy`, `cv2` (OpenCV), and any library from the standard Python library.
2.  **Computer Vision Techniques:** Consider using `cv2` for tasks involving object detection, edge detection, or image filtering.
3.  **Utility Functions:** Write reusable utility functions to improve code modularity and readability.
4.  **Error Handling:** Implement robust error handling to gracefully manage edge cases and invalid inputs.
5.  **Code Clarity:** Write clean, well-documented code with meaningful variable names and comments.

**Part 4: Output Requirements**

1.  **Output Format:**
    *   Begin with a concise paragraph explaining the proposed solution, followed by a Python code section.
    *   You *must* provide a code output representing your best attempt. Do not give up or refuse to produce code.
    *   **The code section must be a single, valid Python code block in markdown fenced code block format and nothing else.**
    *   The main transform function must have the signature `def transform(grid: np.ndarray) -> np.ndarray`.
    *   Document the transformation rule implemented in the docstring of the transform function.
    *   Do not include any `__name__ == "__main__"` block. This will be added later by the user. You are writing a library function.

**Example:**

**Problem:**
Input:
<Diagram>
0 0 1
0 1 0
1 0 0
</Diagram>

Output:
<Diagram>
1 1 1
1 1 1
1 1 1
</Diagram>

**Explanation:**
Replace all 0s with 1s.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all 0s with 1s."""
    return np.where(grid == 0, 1, grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''
SOLVER_PROMPT_3 = '''
You are a world-class expert in solving Abstract Reasoning Corpus (ARC) tasks. Your approach is methodical, creative, and highly effective. You are also a master Python coder, producing elegant, efficient, and well-documented solutions.

Your goal is to analyze a set of input-output examples and devise a Python function that accurately transforms any input grid into its corresponding output grid. The key is to identify a *single, consistent transformation rule* that generalizes across *all* examples. Do not give up until you find a correct solution.

Follow this iterative process:

**Part 1: Initial Analysis and Hypothesis Generation**

1.  **Example Inspection:** Carefully examine the input and output grids for each example. Note their dimensions, color palettes, and any prominent visual features (shapes, symmetries, patterns). Use visualization techniques to aid your analysis.
2.  **Transformation Hypotheses:** Formulate several candidate transformation rules. Start with simpler rules and gradually increase complexity. Consider these categories:
    *   **Color Transformations:** Replacing colors based on specific criteria (e.g., adjacency, frequency). For example, replace all 0s with 1s, or replace the most frequent color with the least frequent color.
    *   **Object Isolation:** Identifying and isolating objects based on color, shape, or position. For example, extract the largest connected component of a certain color, or isolate objects based on their spatial relationships.
    *   **Spatial Operations:** Rotating, reflecting, resizing, or moving objects. For example, rotate the grid by 90 degrees, reflect the grid horizontally or vertically, or resize the grid by a certain factor.
    *   **Pattern Generation:** Replicating or extending existing patterns. For example, repeat a certain pattern across the grid, or generate a new pattern based on the existing patterns.
3.  **Symmetry Analysis:** Identify any symmetries (rotational, reflectional) in the input and output grids. Determine if the transformation preserves or alters these symmetries.

**Part 2: Iterative Testing and Refinement**

1.  **Code Implementation:** Implement your strongest candidate rule as a Python function. The function *must* accept a 2D numpy array as input and return a 2D numpy array as output.
2.  **Rigorous Testing:** Test your code against *all* training examples. A single failure indicates an incorrect rule.
3.  **Feedback Analysis:** If your code fails, carefully analyze the feedback. Identify the specific examples that failed and the nature of the errors. Use print statements to debug intermediate values and verify your assumptions.
4.  **Hypothesis Refinement:** Based on the feedback, refine your transformation rule. This may involve adjusting parameters, adding new conditions, or discarding the rule altogether and starting with a new hypothesis.
5.  **Repeat:** Continue this iterative process of coding, testing, and refining until you find a correct solution. Do not give up until you find a correct solution.

**Part 3: Coding Guidelines**

1.  **Available Libraries:** You can use `numpy`, `cv2` (OpenCV), and any library from the standard Python library.
2.  **Computer Vision Techniques:** Consider using `cv2` for tasks involving object detection, edge detection, or image filtering.
3.  **Utility Functions:** Write reusable utility functions to improve code modularity and readability.
4.  **Error Handling:** Implement robust error handling to gracefully manage edge cases and invalid inputs.
5.  **Code Clarity:** Write clean, well-documented code with meaningful variable names and comments. The code should be as concise as possible.

**Part 4: Output Requirements**

1.  **Output Format:**
    *   Begin with a concise paragraph explaining the proposed solution, followed by a Python code section.
    *   You *must* provide a code output representing your best attempt. Do not give up or refuse to produce code.
    *   **The code section must be a single, valid Python code block in markdown fenced code block format and nothing else.**
    *   The main transform function must have the signature `def transform(grid: np.ndarray) -> np.ndarray`.
    *   Document the transformation rule implemented in the docstring of the transform function.
    *   Do not include any `__name__ == "__main__"` block. This will be added later by the user. You are writing a library function.

**Example 1:**

**Problem:**
Input:
<Diagram>
0 0 1
0 1 0
1 0 0
</Diagram>

Output:
<Diagram>
1 1 1
1 1 1
1 1 1
</Diagram>

**Explanation:**
Replace all 0s with 1s.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all 0s with 1s."""
    return np.where(grid == 0, 1, grid)
```

**Example 2:**

**Problem:**
Input:
<Diagram>
0 0 0
0 1 0
0 0 0
</Diagram>

Output:
<Diagram>
0 1 0
1 1 1
0 1 0
</Diagram>

**Explanation:**
Replace all neighbors of 1 with 1.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all neighbors of 1 with 1."""
    new_grid = grid.copy()
    for i in range(1, grid.shape[0] - 1):
        for j in range(1, grid.shape[1] - 1):
            if grid[i][j] == 1:
                new_grid[i-1][j] = 1
                new_grid[i+1][j] = 1
                new_grid[i][j-1] = 1
                new_grid[i][j+1] = 1
    return new_grid
```

**Example 3:**

**Problem:**
Input:
<Diagram>
1 2 3
4 5 6
7 8 9
</Diagram>

Output:
<Diagram>
9 8 7
6 5 4
3 2 1
</Diagram>

**Explanation:**
Reverse the grid.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Reverse the grid."""
    return np.flip(grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''
FEEDBACK_PROMPT = '''
**EXISTING PARTIAL/INCORRECT SOLUTIONS:**

Following are some of the best, though not completely correct, solutions so far. For each solution, its code, corresponding feedback regarding its output on the example problems, and a numeric score between 0. (worst) and 1. (best) indicating the quality of outputs is also provided. Study these solutions and corresponding feedback and produce a new solution fixing all the issues. Make sure to follow the output format specified earlier.

$$feedback$$
'''

# Replace only the textual prompt templates with the published donor originals.
POETIQ_SOLVER_PROMPTS = {1:SOLVER_PROMPT_1, 2:SOLVER_PROMPT_2, 3:SOLVER_PROMPT_3}
POETIQ_FEEDBACK_PROMPT = FEEDBACK_PROMPT

# A positive runtime test executes before vLLM engine loading (notebook Cell 6).
# Never claim readiness based on a seccomp library import alone.
def poetiq_sandbox_selftest():
    import tempfile
    test_grid=[[2]]
    ok, payload=poetiq_sandbox_run(
        'import numpy as np\ndef transform(grid):\n return np.asarray(grid)', test_grid, 8)
    if not ok or poetiq_grid_from_output(payload)!=test_grid:
        raise RuntimeError('Poetiq sandbox positive test FAILED: '+str(payload)[:500])
    with tempfile.TemporaryDirectory(prefix='e10_root_protected_') as td:
        protected=Path(td)/'no_write'
        protected.write_text('original',encoding='utf-8')
        protected.chmod(0o600)
        attack=f"def transform(grid):\n open({str(protected)!r}, 'w').write('MODIFIED')\n return grid"
        ok,err=poetiq_sandbox_run(attack,test_grid,8)
        if ok or protected.read_text(encoding='utf-8')!='original':
            raise RuntimeError('Poetiq sandbox protected-file write test FAILED')
    for label,attack in (
        ('spawn',"import subprocess\ndef transform(grid):\n subprocess.check_output(['/usr/bin/id'])\n return grid"),
        ('network',"import socket\ndef transform(grid):\n socket.socket()\n return grid"),
    ):
        ok,err=poetiq_sandbox_run(attack,test_grid,8)
        if ok or 'Operation not permitted' not in str(err):
            raise RuntimeError('Poetiq sandbox '+label+' restriction FAILED: '+str(err)[:300])
    return {'ok':True,'mode':'uid_drop+seccomp+rlimits+timeout',
            'filesystem_isolation':'NOT a private mount; world-readable host paths may be readable',
            'verified':['numpy_transform','protected_file_write_blocked','spawn_blocked','network_blocked']}

# Fail closed without losing the cell-0 competition-format fallback.
try:
    E10_SANDBOX_SELFTEST=poetiq_sandbox_selftest()
except BaseException as _e10_sandbox_error:
    E10_SANDBOX_SELFTEST={'ok':False,'mode':'unavailable',
                         'error':type(_e10_sandbox_error).__name__+': '+str(_e10_sandbox_error)[:1200]}
print('[E10] Sandbox self-test:',E10_SANDBOX_SELFTEST,flush=True)


In [ ]:
# ==================================================================== CELL 5 — engine: vLLM 0.32.0 server + in-process shim (E18)
# E18 (user 2026-10-27): S1-60K (LB 34.58) with ONE change, the engine: vLLM 0.32.0 instead of 0.19.0, 20 streams instead
# of 16. Everything the solver sees is unchanged:
#   - the solver (next cell) is byte-identical to S1-60K and calls engine.add_request / step / abort_request,
#     llm.get_metrics(), SamplingParams(...) and RequestOutputKind.DELTA;
#   - this cell provides those names over vLLM 0.32's OpenAI server (streaming /v1/completions with prompt token ids and
#     return_token_ids), so the kernel keeps S1's tokenizer stack (transformers 4.57.6) and the prompts are the same ids.
# Engine flags = S1's ENGINE_CONFIG (FP8 block-128, fp8 KV, MTP 3, prefix caching, FlashInfer if -lcuda links, 4,096
# batched, mem 0.90, custom all-reduce off, language-model-only, generation config 'auto' as LLM()) with max-num-seqs 20.
# vLLM 0.32 server = the VMS launch (Experiment 14, Kaggle 2026-10-26: fp8_mtp3_s20 PASS, 452 / 423 / 281 tok/s at
# 8k / 40k / 90k). Ladder: server (language-model-only) -> server (VMS-exact) -> server (Triton attention) -> S1's own
# in-process vLLM 0.19 engine cell, verbatim (cell 2 still installs vLLM 0.19, as in S1).
import shlex, socket, queue as _queue, signal as _signal, urllib.request as _urlreq, types as _types
import shutil as _shutil
# S1-60K's engine cell (ed530071, cell 6), verbatim: the last fallback below
S1_ENGINE_SRC = "# ==================================================================== CELL 5 \u2014 engine (PIV/V3.1 C1 settings + long context + fp8 KV + MTP)\n# ---- v1.3.4: root cause of Kaggle run 1 (2026-10-05). With fp8 KV, vLLM picks FlashInfer on sm89; FlashInfer JIT-compiled\n# its kernels and then failed to LINK them: `/usr/bin/ld: cannot find -lcuda` (Kaggle ships only libcuda.so.1). Make\n# `-lcuda` linkable BEFORE vLLM starts its worker processes (they inherit LIBRARY_PATH), and prove it with a tiny link.\n# Same fix as dfranzen's Kaggle launcher (libcuda.so symlink), done in a private shim directory.\nimport shutil as _shutil\ndef ensure_libcuda_linkable(candidates=None, workdir=\"/tmp/q27x_libcuda_shim\"):\n    info = {\"needed\": None, \"real\": None, \"shim\": None, \"probe\": \"FAIL\"}\n    try:\n        work = Path(workdir); work.mkdir(parents=True, exist_ok=True)\n        if candidates is None:\n            existing = [Path(d, \"libcuda.so\") for d in (\"/usr/local/cuda/lib64\", \"/usr/local/cuda/lib64/stubs\",\n                                                        \"/usr/lib/x86_64-linux-gnu\", \"/usr/lib64\")]\n            existing = [p for p in existing if p.exists()]\n        else:\n            existing = []\n        if existing:\n            info.update(needed=False, real=str(existing[0]))\n        else:\n            cands = list(candidates) if candidates is not None else []\n            if candidates is None:\n                for d in (\"/usr/local/nvidia/lib64\", \"/usr/local/nvidia/lib\", \"/usr/lib/x86_64-linux-gnu\", \"/usr/lib64\",\n                          \"/usr/local/cuda/compat\"):\n                    try:\n                        cands += sorted(str(p) for p in Path(d).glob(\"libcuda.so.*\") if p.is_file())\n                    except Exception:\n                        pass\n                try:\n                    for line in subprocess.run([\"ldconfig\", \"-p\"], capture_output=True, text=True, timeout=30).stdout.splitlines():\n                        if \"libcuda.so.1\" in line and \"=>\" in line:\n                            cands.append(line.split(\"=>\")[-1].strip())\n                except Exception:\n                    pass\n            real = next((Path(c) for c in cands if Path(c).is_file()), None)\n            if real is None:\n                info[\"reason\"] = \"libcuda.so.1 not found\"\n                return info\n            link = work / \"libcuda.so\"\n            if link.exists() or link.is_symlink():\n                link.unlink()\n            try:\n                link.symlink_to(real.resolve())\n            except OSError:\n                _shutil.copyfile(real, link)\n            old = os.environ.get(\"LIBRARY_PATH\", \"\")\n            os.environ[\"LIBRARY_PATH\"] = f\"{work}:{old}\" if old else str(work)\n            info.update(needed=True, real=str(real), shim=str(link), library_path=os.environ[\"LIBRARY_PATH\"])\n        cxx = _shutil.which(\"c++\") or _shutil.which(\"g++\")\n        if not cxx:\n            info[\"reason\"] = \"no C++ compiler for the link probe\"\n            return info\n        cp = subprocess.run([cxx, \"-shared\", \"-fPIC\", \"-x\", \"c++\", \"-\", \"-lcuda\", \"-o\", str(work / \"probe.so\")],\n                            input='extern \"C\" int q27x_probe() { return 0; }\\n', text=True, capture_output=True,\n                            timeout=120, env=os.environ.copy())\n        info[\"probe\"] = \"PASS\" if cp.returncode == 0 else \"FAIL\"\n        if cp.returncode != 0:\n            info[\"probe_stderr\"] = (cp.stderr or \"\")[-1500:]\n    except Exception as e:\n        info[\"reason\"] = f\"{type(e).__name__}: {e}\"[:500]\n    return info\n\nos.environ.setdefault(\"MAX_JOBS\", \"4\")              # bound FlashInfer JIT build parallelism\nCUDA_LINK = ensure_libcuda_linkable()\nCONFIG[\"cuda_link\"] = CUDA_LINK\nprint(\"CUDA/FlashInfer linker preflight:\", json.dumps(CUDA_LINK))\n\nENGINE_CONFIG = {\n    \"model\": str(MODEL_PATH), \"tensor_parallel_size\": 4, \"distributed_executor_backend\": \"mp\", \"dtype\": \"auto\",\n    \"load_format\": \"safetensors\", \"trust_remote_code\": False, \"language_model_only\": True,\n    \"max_model_len\": MAX_MODEL_LEN, \"max_num_seqs\": MAX_NUM_SEQS, \"max_num_batched_tokens\": MAX_NUM_BATCHED_TOKENS,\n    \"gpu_memory_utilization\": GPU_MEMORY_UTILIZATION, \"enforce_eager\": False, \"disable_custom_all_reduce\": True,\n    \"enable_prefix_caching\": False, \"seed\": SEED,\n    \"disable_log_stats\": False,      # v1.1 (F4): LLM() disables stats by default; we need MTP acceptance / KV usage\n}\nENGINE_INFO = {\"attempts\": [], \"speculative\": None, \"kv_cache_dtype\": None, \"attention_backend\": None, \"load_sec\": None,\n               \"kv_tokens\": None, \"concurrency\": None, \"fatal\": None, \"cuda_link\": CUDA_LINK}\nimport logging\ntry:   # v1.3: keep vLLM's front-end log (stats lines incl. speculative decoding) in the bundle\n    _vllm_log_handler = logging.FileHandler(OUT / \"vllm_frontend.log\", mode=\"w\", encoding=\"utf-8\")\n    _vllm_log_handler.setLevel(logging.INFO)\n    _vllm_log_handler.setFormatter(logging.Formatter(\"%(asctime)s %(levelname)s %(name)s: %(message)s\"))\n    logging.getLogger(\"vllm\").addHandler(_vllm_log_handler)\nexcept Exception as _e:\n    print(\"vLLM log file handler not installed:\", repr(_e)[:200])\nfrom vllm import LLM, SamplingParams\nfrom vllm.sampling_params import RequestOutputKind\nimport vllm\n\ndef _wait_gpu_release(max_sec=float(os.environ.get(\"Q27X_GPU_WAIT_MAX\", 180))):\n    \"\"\"v1.1 (F7) + v1.3: wait until every GPU uses < 1 GB; report whether that actually happened\"\"\"\n    w0 = time.perf_counter(); used = []\n    while time.perf_counter() - w0 < max_sec:\n        try:\n            used = [int(x) for x in subprocess.run([\"nvidia-smi\", \"--query-gpu=memory.used\", \"--format=csv,noheader,nounits\"],\n                                                   capture_output=True, text=True, timeout=20).stdout.split()]\n        except Exception:\n            used = []\n        if used and max(used) < 1024:\n            return {\"released\": True, \"sec\": round(time.perf_counter() - w0, 1), \"last_used_mb\": used}\n        time.sleep(5)\n    return {\"released\": False, \"sec\": round(time.perf_counter() - w0, 1), \"last_used_mb\": used}\n\ndef _kill_child_processes(grace_sec=20):\n    \"\"\"v1.3.3: terminate, then kill, every descendant process of this kernel (vLLM engine core + workers after a failure)\"\"\"\n    try:\n        import psutil\n        kids = psutil.Process(os.getpid()).children(recursive=True)\n        for k in kids:\n            try: k.terminate()\n            except Exception: pass\n        _gone, alive = psutil.wait_procs(kids, timeout=grace_sec)\n        for k in alive:\n            try: k.kill()\n            except Exception: pass\n        psutil.wait_procs(alive, timeout=grace_sec)\n        return {\"found\": len(kids), \"killed_hard\": len(alive)}\n    except Exception as e:\n        return {\"error\": repr(e)[:200]}\n\ndef _engine_smoke(llm_obj):\n    \"\"\"v1.3: a constructor that succeeds is not enough - run a real 16-token decode (fp8 KV / MTP / attention kernels)\"\"\"\n    ids = tokenizer.encode(\"Reply with the digits 1 2 3 4 5 6 7 8:\", add_special_tokens=False)\n    out = llm_obj.generate([{\"prompt_token_ids\": ids}], SamplingParams(max_tokens=16, temperature=0.0, detokenize=False),\n                           use_tqdm=False)\n    toks = list(out[0].outputs[0].token_ids) if out and out[0].outputs else []\n    if not toks:\n        raise RuntimeError(\"engine decode smoke produced no tokens\")\n    return len(toks)\n\n# v1.3.4 ladder: (kv cache dtype, speculative config, attention backend), most capable first. The backend is explicit so\n# vLLM can never fall back to FlashInfer silently. fp8 + FlashInfer only if `-lcuda` links; fp8 + Triton attention keeps\n# the fp8 KV memory saving without FlashInfer's JIT; bf16 + FlashAttention-2 is the PIV/V3.1-proven path.\n# v1.4.1: 4th element = prefix caching. The first fp8 attempt adds it; the next one is the same engine without it (v1.4).\n_LADDER = []\nif KV_CACHE_DTYPE.startswith(\"fp8\"):\n    _first = \"FLASHINFER\" if CUDA_LINK.get(\"probe\") == \"PASS\" else \"TRITON_ATTN\"\n    if ENABLE_PREFIX_CACHING:\n        _LADDER.append((KV_CACHE_DTYPE, SPECULATIVE_CONFIG, _first, True))\n    _LADDER.append((KV_CACHE_DTYPE, SPECULATIVE_CONFIG, _first, False))\n    _LADDER.append((KV_CACHE_DTYPE, SPECULATIVE_CONFIG, \"TRITON_ATTN\", False))\n_LADDER += [(\"auto\", SPECULATIVE_CONFIG, \"FLASH_ATTN\", False), (\"auto\", None, \"FLASH_ATTN\", False)]\n# Do not spend GPU load time when the code-execution sandbox failed startup.\nif not E10_SANDBOX_SELFTEST.get('ok',False):\n    _LADDER = []\n_seen = []\nllm = None\nfor _kv, _spec, _backend, _pc in _LADDER:\n    if (_kv, json.dumps(_spec), _backend, _pc) in _seen:\n        continue\n    _seen.append((_kv, json.dumps(_spec), _backend, _pc))\n    _t0 = time.perf_counter()\n    _kw = dict(ENGINE_CONFIG, attention_config={\"backend\": _backend}, enable_prefix_caching=bool(_pc))\n    if _kv != \"auto\": _kw[\"kv_cache_dtype\"] = _kv\n    if _spec is not None: _kw[\"speculative_config\"] = dict(_spec)\n    _att = {\"kv_cache_dtype\": _kv, \"speculative\": _spec, \"attention_backend\": _backend, \"prefix_caching\": bool(_pc)}\n    print(f\"ENGINE ATTEMPT: kv_cache_dtype={_kv} speculative={_spec} attention_backend={_backend} prefix_caching={_pc}\", flush=True)\n    try:\n        llm = LLM(**_kw)\n        _att[\"smoke_tokens\"] = _engine_smoke(llm)\n        _att.update(ok=True, sec=round(time.perf_counter() - _t0, 1))\n        ENGINE_INFO[\"attempts\"].append(_att)\n        ENGINE_INFO.update(speculative=_spec, kv_cache_dtype=_kv, attention_backend=_backend, prefix_caching=bool(_pc))\n        break\n    except Exception as e:\n        err = f\"{type(e).__name__}: {e}\"[:2000]\n        print(f\"ENGINE ATTEMPT FAILED (kv_cache_dtype={_kv}, speculative={_spec}, backend={_backend}, prefix_caching={_pc}): \"\n              f\"{err[:600]}\", flush=True)\n        try:\n            if llm is not None and hasattr(llm, \"llm_engine\") and hasattr(llm.llm_engine, \"engine_core\"):\n                llm.llm_engine.engine_core.shutdown()\n        except Exception:\n            pass\n        llm = None\n        import gc; gc.collect()\n        # v1.3.3: LLM() itself may raise (run 1), leaving the vLLM workers alive with ~20.7 GB per GPU\n        _att[\"killed_children\"] = _kill_child_processes()\n        _rel = _wait_gpu_release()\n        _att.update(ok=False, error=err, sec=round(time.perf_counter() - _t0, 1), gpu_release=_rel)\n        ENGINE_INFO[\"attempts\"].append(_att)\n        if not _rel[\"released\"]:\n            ENGINE_INFO[\"fatal\"] = \"gpu_memory_not_released_after_failed_engine\"\n            print(\"GPU memory was not released - no further engine attempts\", flush=True)\n            break\nENGINE_READY = llm is not None\nENGINE_INFO[\"load_sec\"] = round(sum(a.get(\"sec\", 0) for a in ENGINE_INFO[\"attempts\"]), 1)\nif not ENGINE_READY:\n    # v1.3: do not raise - the identity submission from cell 0 stays valid, the report explains the failure\n    ENGINE_INFO[\"fatal\"] = ENGINE_INFO[\"fatal\"] or (\"sandbox_selftest_failed\" if not E10_SANDBOX_SELFTEST.get(\"ok\",False) else \"all_engine_attempts_failed\")\n    engine = None; CONCURRENCY = 0\n    print(\"ENGINE INVALID - keeping the identity submission:\", ENGINE_INFO[\"fatal\"], flush=True)\nelse:\n    engine = llm.llm_engine\n    try:   # vLLM copies the engine core's num_gpu_blocks into the front-end cache config at start-up\n        _cc = engine.vllm_config.cache_config\n        ENGINE_INFO[\"kv_tokens\"] = int(_cc.num_gpu_blocks) * int(_cc.block_size) if _cc.num_gpu_blocks else None\n        ENGINE_INFO[\"cache_dtype_resolved\"] = str(getattr(_cc, \"cache_dtype\", None))\n        ENGINE_INFO[\"mamba_cache_mode\"] = str(getattr(_cc, \"mamba_cache_mode\", None))          # v1.4.1: expect 'align'\n        ENGINE_INFO[\"prefix_caching_resolved\"] = bool(getattr(_cc, \"enable_prefix_caching\", False))\n    except Exception as e:\n        ENGINE_INFO[\"kv_tokens\"] = None; ENGINE_INFO[\"kv_error\"] = repr(e)[:300]\n    try:   # what vLLM actually resolved (requested backend is in ENGINE_INFO[\"attention_backend\"])\n        ENGINE_INFO[\"attention_backend_resolved\"] = str(getattr(engine.vllm_config.attention_config, \"backend\", None))\n    except Exception as e:\n        ENGINE_INFO[\"attention_backend_resolved\"] = None; ENGINE_INFO[\"attention_backend_error\"] = repr(e)[:300]\n    CONCURRENCY = MAX_NUM_SEQS\n    if ENGINE_INFO[\"kv_tokens\"]:\n        CONCURRENCY = max(4, min(MAX_NUM_SEQS, ENGINE_INFO[\"kv_tokens\"] // EXPECTED_LIVE_TOKENS_PER_SEQ))\nENGINE_INFO[\"concurrency\"] = int(CONCURRENCY); ENGINE_INFO[\"vllm\"] = vllm.__version__\nprint(\"engine:\", json.dumps(ENGINE_INFO, default=str)[:3000])\n\ndef make_params(max_tokens, seed):\n    return SamplingParams(max_tokens=int(max_tokens), seed=int(seed), detokenize=False, skip_special_tokens=False,\n                          output_kind=RequestOutputKind.DELTA, **SAMPLING)\n\ndef metrics_snapshot():\n    \"\"\"v1.3: vLLM Prometheus snapshot (speculative decoding, preemptions, KV usage, tokens) as plain JSON\"\"\"\n    if not ENGINE_READY:\n        return []\n    try:\n        snap = []\n        for m in llm.get_metrics():\n            name = getattr(m, \"name\", \"\")\n            if not any(k in name for k in (\"spec_decode\", \"preempt\", \"kv_cache\", \"generation_tokens\", \"prompt_tokens\",\n                                           \"num_requests\", \"prefix_cache\")):\n                continue\n            row = {\"name\": name, \"type\": type(m).__name__, \"labels\": getattr(m, \"labels\", None)}\n            for f in (\"value\", \"values\", \"count\", \"sum\"):\n                if hasattr(m, f): row[f] = getattr(m, f)\n            snap.append(row)\n        return snap\n    except Exception as e:\n        return [{\"error\": repr(e)[:300]}]\n"

E18_FAKE = os.environ.get("E18_FAKE_SERVER")          # local CPU harness only: a fake vLLM OpenAI server script
E18_PORT = int(os.environ.get("E18_PORT", 8011))
E18_URL = f"http://127.0.0.1:{E18_PORT}"
E18_STREAM_IDLE_SEC = None      # rev 2 (audits A-M1 / B-L1): no idle timeout; a preempted or waiting request sends no
                                # bytes, as in-process vLLM would simply wait; a dead server is caught by proc.poll()
E18_SERVER = None
E18_SILENCE_SEC = float(os.environ.get("E18_SILENCE_SEC", 600))   # rev 2 (B-M3): smoke v1's longest startup log gap 60 s


def ensure_libcuda_linkable(candidates=None, workdir="/tmp/q27x_libcuda_shim"):
    """S1's cell 5 function, verbatim: make -lcuda linkable for FlashInfer's JIT and prove it with a tiny link."""
    info = {"needed": None, "real": None, "shim": None, "probe": "FAIL"}
    try:
        work = Path(workdir); work.mkdir(parents=True, exist_ok=True)
        if candidates is None:
            existing = [Path(d, "libcuda.so") for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs",
                                                        "/usr/lib/x86_64-linux-gnu", "/usr/lib64")]
            existing = [p for p in existing if p.exists()]
        else:
            existing = []
        if existing:
            info.update(needed=False, real=str(existing[0]))
        else:
            cands = list(candidates) if candidates is not None else []
            if candidates is None:
                for d in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64",
                          "/usr/local/cuda/compat"):
                    try:
                        cands += sorted(str(p) for p in Path(d).glob("libcuda.so.*") if p.is_file())
                    except Exception:
                        pass
                try:
                    for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                        if "libcuda.so.1" in line and "=>" in line:
                            cands.append(line.split("=>")[-1].strip())
                except Exception:
                    pass
            real = next((Path(c) for c in cands if Path(c).is_file()), None)
            if real is None:
                info["reason"] = "libcuda.so.1 not found"
                return info
            link = work / "libcuda.so"
            if link.exists() or link.is_symlink():
                link.unlink()
            try:
                link.symlink_to(real.resolve())
            except OSError:
                _shutil.copyfile(real, link)
            old = os.environ.get("LIBRARY_PATH", "")
            os.environ["LIBRARY_PATH"] = f"{work}:{old}" if old else str(work)
            info.update(needed=True, real=str(real), shim=str(link), library_path=os.environ["LIBRARY_PATH"])
        cxx = _shutil.which("c++") or _shutil.which("g++")
        if not cxx:
            info["reason"] = "no C++ compiler for the link probe"
            return info
        cp = subprocess.run([cxx, "-shared", "-fPIC", "-x", "c++", "-", "-lcuda", "-o", str(work / "probe.so")],
                            input='extern "C" int q27x_probe() { return 0; }\n', text=True, capture_output=True,
                            timeout=120, env=os.environ.copy())
        info["probe"] = "PASS" if cp.returncode == 0 else "FAIL"
        if cp.returncode != 0:
            info["probe_stderr"] = (cp.stderr or "")[-1500:]
    except Exception as e:
        info["reason"] = f"{type(e).__name__}: {e}"[:500]
    return info


def e18_install():
    """The VMS installer: the wheelhouse's install_wheelhouse.py builds /tmp/vllm-venv (--no-deps, '+' restored)."""
    if E18_FAKE:
        return {"fake": True, "launch": [sys.executable, E18_FAKE], "env": dict(os.environ)}
    inst = next(iter(sorted(E18_VLLM_WHEELHOUSE.rglob("install_wheelhouse.py"))), None)
    if inst is None:
        raise FileNotFoundError(f"install_wheelhouse.py not under {E18_VLLM_WHEELHOUSE}: attach the vLLM 0.32.0 wheelhouse")
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, str(inst), "/tmp/vllm-venv"], capture_output=True, text=True, timeout=2400)
    if r.returncode != 0:
        raise RuntimeError("vLLM 0.32 wheelhouse install failed: " + (r.stdout + r.stderr)[-3000:])
    kv = dict(l.split("=", 1) for l in r.stdout.splitlines() if l.startswith(("VENV_PYTHON=", "CUDA_HOME=")))
    env = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "CUDA_PATH")}
    env.update(CUDA_HOME=kv["CUDA_HOME"], PATH=f"/tmp/vllm-venv/bin:{kv['CUDA_HOME']}/bin:{env.get('PATH', '')}",
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", VLLM_NO_USAGE_STATS="1", VLLM_LOGGING_LEVEL="INFO",
               VLLM_CACHE_ROOT="/tmp/vllm-cache", VLLM_ALLOW_LONG_MAX_MODEL_LEN="1")
    return {"sec": round(time.perf_counter() - t0, 1), "venv_python": kv["VENV_PYTHON"],
            "launch": ["/tmp/vllm-venv/bin/vllm", "serve"], "env": env}


def e18_flags(backend, mode):
    """mode 's1': S1's ENGINE_CONFIG as vllm serve flags (incl. language_model_only, trust_remote_code False) with the
    20 seqs of cell 1. mode 'vms': the exact flag set of the Kaggle-run VMS arm fp8_mtp3_s20 (2026-10-26)."""
    f = [str(MODEL_PATH), "--tensor-parallel-size", "4", "--kv-cache-dtype", KV_CACHE_DTYPE,
         "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_NUM_SEQS),
         "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS), "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
         "--disable-custom-all-reduce", "--seed", str(SEED), "--host", "127.0.0.1", "--port", str(E18_PORT),
         "--served-model-name", "q27", "--attention-backend", backend]
    if mode == "s1":
        f += ["--distributed-executor-backend", "mp", "--dtype", "auto", "--load-format", "safetensors",
              "--language-model-only"]
    else:
        f += ["--trust-remote-code"]
    f += ["--enable-prefix-caching"] if ENABLE_PREFIX_CACHING else ["--no-enable-prefix-caching"]
    if SPECULATIVE_CONFIG is not None:
        f += ["--speculative-config", json.dumps(SPECULATIVE_CONFIG)]
    return f


class E18Server:
    def __init__(self, name, cmd, env):
        self.name, self.cmd, self.env, self.proc = name, list(cmd), dict(env), None
        self.log_path = OUT / f"vllm_server_{name}.log"

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     start_new_session=True, cwd=None if E18_FAKE else "/tmp")

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout, silence=E18_SILENCE_SEC):
        """rev 2: ready = /health answers AND this server's own "GPU KV cache size" line is in its log (B-L8); a launch
        whose log stops growing for `silence` seconds is a stall (B-M3)."""
        last, last_t = -1, time.perf_counter()
        while time.perf_counter() - self.t0 < timeout:
            if not self.alive():
                return False, f"exited rc={self.proc.returncode}"
            try:
                _urlreq.urlopen(E18_URL + "/health", timeout=10).read()
                if "GPU KV cache size" in self.log_text():
                    return True, f"ready in {time.perf_counter() - self.t0:.0f}s"
            except Exception:
                pass
            sz = self.log_path.stat().st_size if self.log_path.exists() else 0
            if sz != last:
                last, last_t = sz, time.perf_counter()
            elif time.perf_counter() - last_t > silence:
                return False, f"stall ({silence:.0f} s of log silence)"
            time.sleep(1 if E18_FAKE else 5)
        return False, "timeout"

    def kill(self, fast=False):
        """rev 2 (B-M2): SIGTERM to the process group, a grace for the API server, then SIGKILL to the group ALWAYS
        (engine core / workers stuck in CUDA or NCCL can outlive the API server)."""
        if self.proc is None:
            return
        grp = hasattr(os, "killpg")
        try:
            os.killpg(self.proc.pid, _signal.SIGTERM) if grp else self.proc.terminate()
        except Exception:
            pass
        try:
            self.proc.wait(timeout=0.1 if fast else 30)
        except Exception:
            pass
        try:
            os.killpg(self.proc.pid, getattr(_signal, "SIGKILL", _signal.SIGTERM)) if grp else self.proc.kill()
        except Exception:
            pass
        try:
            self.proc.wait(timeout=30)
        except Exception:
            pass
        try:
            self.log_f.close()
        except Exception:
            pass


def _e18_kill_gpu_pids():
    """rev 2 (B-M2): kill every process still holding a GPU compute context (never this kernel)."""
    if E18_FAKE:
        return []
    killed = []
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], capture_output=True,
                             text=True, timeout=20).stdout
        for tok in out.split():
            if tok.isdigit() and int(tok) != os.getpid():
                try:
                    os.kill(int(tok), getattr(_signal, "SIGKILL", _signal.SIGTERM)); killed.append(int(tok))
                except Exception:
                    pass
    except Exception:
        pass
    return killed


# ---- the shim: S1's vLLM engine calls on top of the vLLM 0.32 OpenAI server
class SamplingParams:
    """Stand-in for vllm.SamplingParams: keeps the fields the solver sets (max_tokens, seed, temperature, top_p, top_k,
    detokenize, skip_special_tokens, output_kind)."""
    def __init__(self, **kw):
        self.__dict__.update(kw)


RequestOutputKind = _types.SimpleNamespace(DELTA="delta", CUMULATIVE="cumulative", FINAL_ONLY="final_only")


class _E18Completion:
    __slots__ = ("token_ids", "finish_reason")

    def __init__(self, ids, reason):
        self.token_ids, self.finish_reason = ids, reason


class _E18Output:
    __slots__ = ("request_id", "outputs", "finished")

    def __init__(self, rid, ids, reason):
        self.request_id, self.outputs, self.finished = rid, [_E18Completion(ids, reason)], reason is not None


class E18Engine:
    """add_request starts one streaming /v1/completions per request in a thread; step() returns the token deltas that
    arrived (merged per request, order kept), raises when the server died; abort_request closes the streams (vLLM aborts
    a request whose client disconnects) and drops their undelivered deltas, as an in-process abort does."""

    def __init__(self, server):
        self.server, self.q, self.aborted, self.streams = server, _queue.Queue(), set(), {}
        self.lock = threading.Lock()
        self.stats = {"added": 0, "finished": 0, "by_reason": {}, "errors": [], "completion_tokens": 0, "aborts": 0,
                      "missing_token_ids": 0, "relaunches": []}
        self.req_log = OUT / "vllm_requests.jsonl"

    def _ensure_server(self):
        """rev 2 (B-M1): the server died -> relaunch it once with the same command, while >= 25 min remain before the
        hard stop. The streams of the dead server end as 'error' by themselves (connection errors): the solver closes
        those units as in S1 (phase 2 / requeue) on the new server. Otherwise raise, as S1's in-process engine did."""
        global E18_SERVER
        if self.server.alive():
            return
        rc = self.server.proc.returncode
        if self.stats["relaunches"] or time.perf_counter() > HARD_STOP - 25 * 60:
            raise RuntimeError(f"vLLM server exited (rc={rc}); see {self.server.log_path}")
        t0 = time.perf_counter()
        rec = {"rc": rc, "t_sec": round(t0 - NOTEBOOK_T0, 1), "old_log": str(self.server.log_path)}
        print(f"E18: vLLM server exited (rc={rc}); relaunching once", flush=True)
        self.server.kill(fast=True)
        rec["gpu_pids_killed"] = _e18_kill_gpu_pids()
        rec["gpu_release"] = _wait_gpu_release() if not E18_FAKE else {"released": True, "fake": True}
        new = E18Server(self.server.name + "_relaunch", self.server.cmd, self.server.env)
        new.start()
        ok, msg = new.wait_ready(20 * 60)
        rec["ready"] = msg
        if ok:
            E18_SERVER, self.server = new, new
            try:
                rec["smoke"] = e18_smoke()
            except Exception as e:
                ok, rec["smoke_error"] = False, f"{type(e).__name__}: {e}"[:400]
        rec["sec"] = round(time.perf_counter() - t0, 1)
        self.stats["relaunches"].append(rec)
        print("E18 relaunch:", json.dumps(rec, default=str)[:800], flush=True)
        if not ok:
            new.kill(fast=True)
            raise RuntimeError(f"vLLM server relaunch failed: {msg} {rec.get('smoke_error', '')}")

    def add_request(self, rid, prompt, params):
        self._ensure_server()
        body = {"model": "q27", "prompt": [int(x) for x in prompt["prompt_token_ids"]], "max_tokens": int(params.max_tokens),
                "temperature": float(params.temperature), "top_p": float(params.top_p), "top_k": int(params.top_k),
                "stream": True, "return_token_ids": True, "skip_special_tokens": False}
        if getattr(params, "seed", None) is not None:
            body["seed"] = int(params.seed)
        st = {"rid": rid, "resp": None, "halt": False}
        with self.lock:
            self.aborted.discard(rid)
            self.streams[rid] = st
            self.stats["added"] += 1
        threading.Thread(target=self._stream, args=(st, json.dumps(body).encode(), time.perf_counter()), daemon=True).start()

    def _stream(self, st, body, t0):
        rid, reason, err, n = st["rid"], None, None, 0
        try:
            req = _urlreq.Request(E18_URL + "/v1/completions", data=body, method="POST",
                                  headers={"Content-Type": "application/json"})
            st["resp"] = _urlreq.urlopen(req, timeout=E18_STREAM_IDLE_SEC)
            if st["halt"]:                       # rev 2 (A-L3 / B-L2): aborted while the request was being opened
                raise ConnectionAbortedError("aborted before the first chunk")
            for raw in st["resp"]:
                if st["halt"]:
                    break
                line = raw.strip()
                if not line.startswith(b"data:"):
                    continue
                data = line[5:].strip()
                if data == b"[DONE]":
                    break
                d = json.loads(data)
                if "error" in d or d.get("object") == "error":
                    err = ("stream error: " + json.dumps(d.get("error") or d.get("message") or d))[:300]
                    break
                for c in d.get("choices") or []:
                    ids = c.get("token_ids")
                    if ids is None:
                        if c.get("text"):
                            with self.lock:
                                self.stats["missing_token_ids"] += 1
                        ids = []
                    ids = [int(x) for x in ids]
                    fr = c.get("finish_reason")
                    if fr is not None and reason is None:
                        reason = str(fr)
                    n += len(ids)
                    if ids or fr is not None:
                        self.q.put((rid, ids, None if fr is None else str(fr)))
                if reason is not None:
                    break
        except Exception as e:
            if not st["halt"]:
                err = f"{type(e).__name__}: {e}"[:300]
        finally:                                 # rev 2 (A-L3 / B-L1): close on every exit (vLLM aborts on disconnect)
            try:
                if st["resp"] is not None:
                    st["resp"].close()
            except Exception:
                pass
        if reason is None and not st["halt"]:
            reason = "error"
            self.q.put((rid, [], "error"))
        with self.lock:
            s = self.stats
            s["finished"] += 1
            rr = "abort" if st["halt"] else reason
            s["by_reason"][rr] = s["by_reason"].get(rr, 0) + 1
            s["completion_tokens"] += n
            if err and len(s["errors"]) < 50:
                s["errors"].append(f"{rid}: {err}")
            self.streams.pop(rid, None)
            try:
                with open(self.req_log, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"rid": rid, "finish": rr, "tokens": n, "sec": round(time.perf_counter() - t0, 1),
                                        "error": err}) + "\n")
            except Exception:
                pass

    def step(self):
        self._ensure_server()
        items = []
        try:
            items.append(self.q.get(timeout=0.05))
        except _queue.Empty:
            return []
        while True:
            try:
                items.append(self.q.get_nowait())
            except _queue.Empty:
                break
        merged = {}
        with self.lock:
            aborted = set(self.aborted)
        for rid, ids, reason in items:
            if rid in aborted:
                continue
            m = merged.setdefault(rid, [[], None])
            if m[1] is None:
                m[0].extend(ids)
                m[1] = reason
        return [_E18Output(rid, ids, reason) for rid, (ids, reason) in merged.items()]

    def abort_request(self, ids):
        for rid in list(ids):
            with self.lock:
                self.aborted.add(rid)
                st = self.streams.get(rid)
                self.stats["aborts"] += 1
            if st is None:
                continue
            st["halt"] = True
            try:   # shut the socket first so a reader blocked in recv returns, then close (vLLM aborts on disconnect)
                st["resp"].fp.raw._sock.shutdown(socket.SHUT_RDWR)
            except Exception:
                pass
            try:
                if st["resp"] is not None:
                    st["resp"].close()
            except Exception:
                pass


_E18_PROM_RE = re.compile(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)")


class E18LLM:
    """llm.get_metrics(): the server's Prometheus metrics as objects with name / labels / value (what S1 reads)."""
    def get_metrics(self):
        txt = _urlreq.urlopen(E18_URL + "/metrics", timeout=2).read().decode("utf-8", "replace")   # rev 2 (B-L9)
        out = []
        for line in txt.splitlines():
            if not line or line.startswith("#"):
                continue
            m = _E18_PROM_RE.match(line)
            if not m or m.group(1).endswith(("_bucket", "_created")):
                continue
            try:
                v = float(m.group(3))
            except ValueError:
                continue
            out.append(_types.SimpleNamespace(name=m.group(1), labels=m.group(2), value=v))
        return out


def e18_smoke():
    """A real 16-token greedy decode through the shim (fp8 KV / MTP / attention kernels), as S1's _engine_smoke."""
    ids = tokenizer.encode("Reply with the digits 1 2 3 4 5 6 7 8:", add_special_tokens=False)
    eng = E18Engine(E18_SERVER)
    eng.add_request("e18-smoke", {"prompt_token_ids": ids}, SamplingParams(max_tokens=16, temperature=0.0, top_p=1.0,
                                                                         top_k=1, seed=SEED))
    toks, reason, t_end = [], None, time.perf_counter() + 120         # rev 2 (B-M3): smoke v1 took ~1 s
    while reason is None and time.perf_counter() < t_end:
        for o in eng.step():
            toks.extend(o.outputs[0].token_ids)
            if o.finished:
                reason = o.outputs[0].finish_reason
    if not toks or reason in (None, "error"):
        raise RuntimeError(f"engine decode smoke failed: {len(toks)} tokens, finish {reason}, errors {eng.stats['errors'][:2]}")
    return {"tokens": len(toks), "finish": reason, "text": tokenizer.decode(toks)[:80]}


def _wait_gpu_release(max_sec=float(os.environ.get("Q27X_GPU_WAIT_MAX", 180))):
    """S1's cell 5 function, verbatim."""
    w0 = time.perf_counter(); used = []
    while time.perf_counter() - w0 < max_sec:
        try:
            used = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                                   capture_output=True, text=True, timeout=20).stdout.split()]
        except Exception:
            used = []
        if used and max(used) < 1024:
            return {"released": True, "sec": round(time.perf_counter() - w0, 1), "last_used_mb": used}
        time.sleep(5)
    return {"released": False, "sec": round(time.perf_counter() - w0, 1), "last_used_mb": used}


os.environ.setdefault("MAX_JOBS", "4")
CUDA_LINK = ensure_libcuda_linkable() if not E18_FAKE else {"probe": "PASS", "fake": True}
CONFIG["cuda_link"] = CUDA_LINK
print("CUDA/FlashInfer linker preflight:", json.dumps(CUDA_LINK))
ENGINE_INFO = {"engine": "vllm-0.32.0-server", "attempts": [], "speculative": None, "kv_cache_dtype": None,
               "attention_backend": None, "load_sec": None, "kv_tokens": None, "concurrency": None, "fatal": None,
               "cuda_link": CUDA_LINK, "install": None}
llm = engine = None
ENGINE_READY = False
_e18_first = "FLASHINFER" if CUDA_LINK.get("probe") == "PASS" else "TRITON_ATTN"
_E18_LADDER = [("s1", _e18_first, "s1"), ("vms", _e18_first, "vms"), ("vms_triton", "TRITON_ATTN", "vms")]
if _e18_first == "TRITON_ATTN":
    _E18_LADDER = _E18_LADDER[:2]
if not E10_SANDBOX_SELFTEST.get('ok', False):
    _E18_LADDER = []
_t_engine = time.perf_counter()
try:
    if _E18_LADDER:
        _inst = e18_install()
        ENGINE_INFO["install"] = {k: v for k, v in _inst.items() if k != "env"}
        print("vLLM 0.32 install:", json.dumps(ENGINE_INFO["install"]), flush=True)
    for _i, (_name, _backend, _mode) in enumerate(_E18_LADDER):
        _t0 = time.perf_counter()
        _att = {"name": _name, "attention_backend": _backend, "flag_mode": _mode, "flags": e18_flags(_backend, _mode)}
        print(f"ENGINE ATTEMPT {_name}: vllm serve {' '.join(_att['flags'])}", flush=True)
        E18_SERVER = E18Server(_name, _inst["launch"] + _att["flags"], _inst["env"])
        E18_SERVER.start()
        _ok, _msg = E18_SERVER.wait_ready((30 if _i == 0 else 15) * 60)       # rev 2 (B-M3): 20 -> 15 min
        _att["ready"] = _msg
        if _ok:
            try:
                _att["smoke"] = e18_smoke()
            except Exception as _se:
                _ok, _att["smoke_error"] = False, f"{type(_se).__name__}: {_se}"[:600]
        _att["sec"] = round(time.perf_counter() - _t0, 1)
        if _ok:
            _att["ok"] = True
            ENGINE_INFO["attempts"].append(_att)
            break
        _att["ok"] = False
        _att["log_tail"] = E18_SERVER.log_text()[-2500:]
        print(f"ENGINE ATTEMPT {_name} FAILED: {_msg} {_att.get('smoke_error', '')} | {_att['log_tail'][-800:]}", flush=True)
        E18_SERVER.kill()
        _att["gpu_pids_killed"] = _e18_kill_gpu_pids()                        # rev 2 (B-M2)
        _att["gpu_release"] = _wait_gpu_release() if not E18_FAKE else {"released": True, "fake": True}
        ENGINE_INFO["attempts"].append(_att)
        E18_SERVER = None
        if not _att["gpu_release"]["released"]:
            ENGINE_INFO["fatal"] = "gpu_memory_not_released_after_failed_engine"
            break
    if E18_SERVER is not None:
        engine, llm, ENGINE_READY = E18Engine(E18_SERVER), E18LLM(), True
except Exception as _e:
    ENGINE_INFO["attempts"].append({"name": "install/launch", "ok": False, "error": f"{type(_e).__name__}: {_e}"[:2000]})
    print("vLLM 0.32 engine error:", ENGINE_INFO["attempts"][-1]["error"][:1500], flush=True)
    if E18_SERVER is not None:
        E18_SERVER.kill()
        E18_SERVER = None
    _e18_kill_gpu_pids()                                                      # rev 2 (B-L3)
    _rel = _wait_gpu_release() if not E18_FAKE else {"released": True, "fake": True}
    if not _rel["released"]:
        ENGINE_INFO["fatal"] = "gpu_memory_not_released_after_failed_engine"

if ENGINE_READY:
    _log = E18_SERVER.log_text()
    _kv = re.findall(r"GPU KV cache size: ([\d,]+) tokens", _log)
    _ver = re.findall(r"version (\d+\.\d+\.\d+)", _log)
    _a = ENGINE_INFO["attempts"][-1]
    ENGINE_INFO.update(speculative=SPECULATIVE_CONFIG, kv_cache_dtype=KV_CACHE_DTYPE, attention_backend=_a["attention_backend"],
                       prefix_caching=ENABLE_PREFIX_CACHING, flag_mode=_a["flag_mode"],
                       kv_tokens=int(_kv[-1].replace(",", "")) if _kv else None, vllm=_ver[0] if _ver else None,
                       server_log=str(E18_SERVER.log_path))
    CONCURRENCY = MAX_NUM_SEQS
    if ENGINE_INFO["kv_tokens"]:
        CONCURRENCY = max(4, min(MAX_NUM_SEQS, ENGINE_INFO["kv_tokens"] // EXPECTED_LIVE_TOKENS_PER_SEQ))
    ENGINE_INFO["load_sec"] = round(time.perf_counter() - _t_engine, 1)
    ENGINE_INFO["concurrency"] = int(CONCURRENCY)

    def make_params(max_tokens, seed):
        return SamplingParams(max_tokens=int(max_tokens), seed=int(seed), detokenize=False, skip_special_tokens=False,
                              output_kind=RequestOutputKind.DELTA, **SAMPLING)

    def metrics_snapshot():
        """S1's metric filter over the server's /metrics."""
        if not ENGINE_READY:
            return []
        try:
            snap = []
            for m in llm.get_metrics():
                name = getattr(m, "name", "")
                if not any(k in name for k in ("spec_decode", "preempt", "kv_cache", "generation_tokens", "prompt_tokens",
                                               "num_requests", "prefix_cache")):
                    continue
                snap.append({"name": name, "type": "prometheus", "labels": m.labels, "value": m.value})
            return snap
        except Exception as e:
            return [{"error": repr(e)[:300]}]
    print("engine:", json.dumps(ENGINE_INFO, default=str)[:3000], flush=True)
elif E10_SANDBOX_SELFTEST.get('ok', False) and ENGINE_INFO.get("fatal") is None and os.environ.get("E18_NO_FALLBACK") != "1":
    # every vLLM 0.32 attempt failed: S1-60K's own engine cell (in-process vLLM 0.19), verbatim
    print("E18: vLLM 0.32 unavailable -> S1-60K's vLLM 0.19 engine (in-process)", flush=True)
    _e18_attempts = list(ENGINE_INFO["attempts"])
    MAX_NUM_SEQS = 16                       # rev 2 (B-L5): S1's own engine with S1's 16 sequences
    try:
        exec(compile(S1_ENGINE_SRC, "<S1 engine cell>", "exec"), globals())
        ENGINE_INFO["engine"] = "vllm-0.19.0-in-process (E18 fallback)"
    except BaseException as _fe:
        ENGINE_READY = False; engine = None; CONCURRENCY = 0
        ENGINE_INFO["fatal"] = f"fallback vLLM 0.19 failed: {type(_fe).__name__}: {_fe}"[:600]
        ENGINE_INFO["concurrency"] = 0
        print("ENGINE INVALID - keeping the identity submission:", ENGINE_INFO["fatal"], flush=True)
    ENGINE_INFO["e18_attempts"] = _e18_attempts
else:
    ENGINE_INFO["fatal"] = ENGINE_INFO["fatal"] or ("sandbox_selftest_failed" if not E10_SANDBOX_SELFTEST.get("ok", False)
                                                    else "all_engine_attempts_failed")
    engine = None; CONCURRENCY = 0
    ENGINE_INFO["concurrency"] = 0
    print("ENGINE INVALID - keeping the identity submission:", ENGINE_INFO["fatal"], flush=True)


In [ ]:
# E10 — Poetiq iterative program-synthesis solver on the existing Q27 86K vLLM engine.
import concurrent.futures as futures
from collections import deque

# Reinstated from the original E10 base; required at the first engine step.
LOG_EVERY_SEC = 120.0

E10_RECORDS = []
E10_RESULT_PATH = OUT / 'poetiq_iterations.jsonl'
E10_PROGRAM_DIR = OUT / 'poetiq_programs'; E10_PROGRAM_DIR.mkdir(parents=True,exist_ok=True)
E10_THINKING_DIR = OUT / 'poetiq_thinking'; E10_THINKING_DIR.mkdir(parents=True,exist_ok=True)   # R4T: full thinking per iteration
E10_FINAL = {} # task -> list of Kaggle attempt pairs; EVAL written separately
E10_STATES = {}
E10_QUEUE = deque()
E10_ACTIVE = {} # (tid, expert) -> running phase state
E10_VERIFY = {} # future -> (tid, expert, result)
E10_TOTAL_GEN = 0
E10_SCHED = {'started':time.perf_counter(),'requests':0,'iterations_finished':0,'programs_parsed':0,
             'train_passed':0,'expert_count':POETIQ_NUM_EXPERTS,'max_iterations':POETIQ_MAX_ITERATIONS,
             'max_active':0,'aborted':0,'steps':0,'skipped_context':0,'deadline_not_started':0,
             'sandbox_verifications':0,'voting_updates':0,'feedback_trims':0,'sandbox_startup':E10_SANDBOX_SELFTEST,
             'kv_gate_blocks':0,'kv_gate_last':None,'unit_errors':[]}

# S1 (Q): iteration 1 in ORDER2 order ([>=3 train] + [1 test], ties keep the sorted-id order). Seeds still use the
# sorted-id index. Failed tasks go to the back (unchanged), so every task gets iteration 1 before any iteration 2.
def e10_order2_score(tid):
    _t=CHALLENGES[tid]
    return int(len(_t['train'])>=3)+int(len(_t['test'])==1)
_E10_INDEX={t:i for i,t in enumerate(TASK_IDS)}
E10_ORDER=sorted(TASK_IDS,key=lambda t:(-e10_order2_score(t),_E10_INDEX[t]))
for _tid in E10_ORDER:
    _task_idx=_E10_INDEX[_tid]
    for _expert in range(POETIQ_NUM_EXPERTS):
        _key=(_tid,_expert)
        E10_STATES[_key] = {'task_id':_tid,'expert':_expert,'task_index':_task_idx,
                            'iteration':0,'finished':False,'history':[], 'candidate_history':[], 'best':None,
                            'best_score':-1.0,'last_error':None,'status':'queued'}
        E10_QUEUE.append(_key)

def e10_render_request(st, max_feedback=None):
    it=st['iteration']
    idx = POETIQ_SOLVER_VARIANTS[st['expert'] % len(POETIQ_SOLVER_VARIANTS)]
    limit = POETIQ_MAX_SOLUTIONS if max_feedback is None else max_feedback
    msg = poetiq_build_message(CHALLENGES[st['task_id']],it,st['expert'],st['history'],
           variant=idx,selection_probability=POETIQ_SELECTION_PROB,
           max_solutions=limit,improving_order=POETIQ_IMPROVING_ORDER,
           shuffle=POETIQ_SHUFFLE_EXAMPLES,seed=POETIQ_SEED+st['task_index']*1000+st['expert']*100)
    chat=[{'role':'user','content':msg}]
    return tokenizer.apply_chat_template(chat,tokenize=False,add_generation_prompt=True,
                        enable_thinking=True,reasoning_effort=REASONING_EFFORT)

def e10_seed(st, phase):
    return SEED + st['task_index']*1000 + st['expert']*100 + st['iteration']*2 + int(phase)

def e10_make_params(cap,seed):
    return SamplingParams(max_tokens=int(cap),seed=int(seed),
                          detokenize=False,skip_special_tokens=False,
                          output_kind=RequestOutputKind.DELTA,
                          temperature=POETIQ_SOLVER_TEMPERATURE,
                          top_p=SAMPLING['top_p'],top_k=SAMPLING['top_k'])

def e10_req(st,phase):
    return f"{st['task_id']}~e{st['expert']}~i{st['iteration']}|{phase}"

def e10_admit(key):
    st=E10_STATES[key]
    rendered=e10_render_request(st)
    ids=tokenizer.encode(rendered,add_special_tokens=False)
    cap=phase1_cap(len(ids)) # preserve original 86K configured allowance
    # Never admit below the user's 49,152 minimum. Remove only low-score
    # historical feedback examples, retaining Poetiq's best-score ordering.
    # Entire history remains intact for future iterations and voting.
    trimmed_feedback=False
    if cap < 49152 and st['history']:
        for keep in range(min(POETIQ_MAX_SOLUTIONS,len(st['history']))-1,-1,-1):
            rendered=e10_render_request(st,max_feedback=keep)
            ids=tokenizer.encode(rendered,add_special_tokens=False)
            cap=phase1_cap(len(ids))
            if cap >= 49152:
                trimmed_feedback=True
                break
    if trimmed_feedback:
        E10_SCHED['feedback_trims']+=1
    if cap < 49152:
        st['finished']=True; st['status']='insufficient_context'
        st['last_error']=f'prompt {len(ids)} tokens leaves {cap} thinking room'
        E10_SCHED['skipped_context']+=1
        return 'skipped'
    req={'key':key,'uid':e10_req(st,1),'phase':1,'prompt_ids':ids,'p1_cap':cap,
          'p1_ids':[],'p2_ids':[],'p2_prefix':[],'p1_finish':None,'p2_finish':None,
          't_admit':time.perf_counter(),'mode':None}
    # Commit active state only when the vLLM request is registered.
    engine.add_request(e10_req(st,1),{'prompt_token_ids':list(ids)},e10_make_params(cap,e10_seed(st,1)))
    E10_ACTIVE[key]=req;st['status']='running'
    E10_SCHED['requests']+=1
    return 'admitted'

def e10_start_phase2(req,answer_cap=ANSWER_MAX_TOKENS):
    st=E10_STATES[req['key']]
    closed=THINK_CLOSE_ID in req['p1_ids']
    if not closed:
        while req['p1_ids'] and req['p1_ids'][-1] in EOS_IDS: req['p1_ids'].pop()
    pref=list(req['prompt_ids'])+list(req['p1_ids'])+([] if closed else list(STOP_IDS))
    cap=min(int(answer_cap),MAX_MODEL_LEN-len(pref)-8)
    if cap<64:return False
    # Do not advance phase before successful registration.
    engine.add_request(e10_req(st,2),{'prompt_token_ids':pref},e10_make_params(cap,e10_seed(st,2)))
    req['phase']=2;req['p2_prefix']=[] if closed else list(STOP_IDS)
    req['mode']='continue' if closed else 'forced_close'
    return True

def e10_answer_text(req):
    ids=list(req['p1_ids'])+list(req['p2_prefix'])+list(req['p2_ids'])
    if THINK_CLOSE_ID not in ids:
        return '',False
    idx=ids.index(THINK_CLOSE_ID)
    return tokenizer.decode(ids[idx+1:],skip_special_tokens=True), True

def e10_after_decode(req,reason,pool):
    key=req['key'];st=E10_STATES[key]
    E10_ACTIVE.pop(key,None)
    text, closed=e10_answer_text(req)
    code=poetiq_parse_code(text)
    try:   # R4T: save the complete generation (thinking + stop text + answer); logging only
        _full=tokenizer.decode(list(req['p1_ids'])+list(req['p2_prefix'])+list(req['p2_ids']),skip_special_tokens=False)
        _head=(f"# task={st['task_id']} expert={st['expert']} iteration={st['iteration']+1} mode={req['mode'] or 'natural'} "
               f"finish={reason} prompt_tokens={len(req['prompt_ids'])} p1_tokens={len(req['p1_ids'])} p2_tokens={len(req['p2_ids'])}\n")
        (E10_THINKING_DIR/f"{st['task_id']}_e{st['expert']}_i{st['iteration']+1}.txt").write_text(_head+_full,encoding='utf-8')
    except Exception as _exc:
        print('[E10] thinking save failed:',st['task_id'],repr(_exc)[:200],flush=True)
    E10_SCHED['iterations_finished']+=1
    base_rec={'task_id':st['task_id'],'expert':st['expert'],'iteration':st['iteration']+1,
              'mode':req['mode'] or 'natural','finish':reason,
              'p1_tokens':len(req['p1_ids']),'p2_tokens':len(req['p2_ids']),
              'prompt_tokens':len(req['prompt_ids']),'p1_cap':req['p1_cap'],
              'sec':round(time.perf_counter()-req['t_admit'],2),'has_python':bool(code)}
    if code:
        E10_SCHED['programs_parsed']+=1
        try:
            (E10_PROGRAM_DIR / f"{st['task_id']}_e{st['expert']}_i{st['iteration']+1}.py").write_text(code,encoding='utf-8')
        except Exception:pass
        try:
            future=pool.submit(poetiq_verify_code,code,CHALLENGES[st['task_id']],POETIQ_SANDBOX_TIMEOUT)
            E10_VERIFY[future]=(key,code,base_rec)
            st['status']='verifying'
            return
        except Exception as exc:
            base_rec['verification_submit_error']=repr(exc)
    st['last_error']='missing code block' if not code else 'verification scheduling failed'
    e10_commit_iteration(key,None,None,base_rec)

def e10_commit_iteration(key,code,verification,base_rec):
    st=E10_STATES[key]
    result=None
    if verification:
        result=dict(verification)
        result.update({'task_id':st['task_id'],'expert':st['expert'],
                       'iteration':st['iteration']+1,'code':code})
        feedback=result['feedback'];score=result['score']
        st['history'].append({'code':code,'feedback':feedback,'score':score})
        # All executed programs stay available for a second diverse attempt.
        # Best/expert voting remains the original primary selection mechanism.
        st['candidate_history'].append(result)
        usable=bool(result.get('test_usable',False))
        # S1 (F3): a train-passing program stays eligible as best even if one test input fails; attempts are then
        # picked per test input from valid grids (R5 dropped it and lost every query of the task).
        if (usable or result['passed']) and score >= st['best_score']:
            st['best_score']=score;st['best']=result
        if result['passed'] and usable:
            st['finished']=True;st['status']='train_verified'
            E10_SCHED['train_passed']+=1
        elif result['passed'] and not usable:
            st['status']='train_pass_test_unusable'
    base_rec['score']=result['score'] if result else None
    base_rec['train_passed']=bool(result and result['passed'])
    E10_RECORDS.append(base_rec)
    with open(E10_RESULT_PATH,'a',encoding='utf-8') as log:log.write(json.dumps(base_rec)+'\n')
    st['iteration']+=1
    if st['iteration']>=POETIQ_MAX_ITERATIONS:st['finished']=True
    if not st['finished']:
        st['status']='queued';E10_QUEUE.append(key)
    elif st['status']!='train_verified':st['status']='iterations_exhausted'
    e10_update_task_submission(st['task_id'])
    print(f"[E10] {st['task_id']} expert={st['expert']} iter={st['iteration']} "
          f"score={base_rec['score']} passed={base_rec['train_passed']} status={st['status']}",flush=True)

def e10_poll_verifications():
    for fut in list(E10_VERIFY):
        if not fut.done():continue
        key,code,rec=E10_VERIFY.pop(fut)
        try:
            result=fut.result()
        except Exception as err:
            result=None;rec['verify_error']=repr(err)[:1000]
        E10_SCHED['sandbox_verifications']+=1
        e10_commit_iteration(key,code,result,rec)

def e10_rank_task(tid):
    # Preserve Poetiq's main expert-best voting ranking. Append earlier valid
    # (possibly partial-train) program histories only as extra distinct
    # candidates for attempt_2, never ahead of the donor's top experts.
    expert_bests=[s['best'] for (task,_),s in E10_STATES.items() if task==tid and s['best']]
    voted=poetiq_vote(expert_bests,use_new_voting=POETIQ_USE_NEW_VOTING,
          count_failed_matches=POETIQ_COUNT_FAILED_MATCHES,
          iters_tiebreak=POETIQ_ITERS_TIEBREAK,
          low_to_high_iters=POETIQ_LOW_TO_HIGH_ITERS)
    already={id(r) for r in voted}
    history=[r for (task,_),s in E10_STATES.items() if task==tid
             for r in s['candidate_history']
             if id(r) not in already]   # S1 (F3): keep partly usable candidates; grids are checked per test input
    if history:
        # Same donor voting/soft-score semantics for historical candidates;
        # no synthetic identity/mirror candidates are introduced.
        voted.extend(poetiq_vote(history,use_new_voting=POETIQ_USE_NEW_VOTING,
            count_failed_matches=POETIQ_COUNT_FAILED_MATCHES,
            iters_tiebreak=POETIQ_ITERS_TIEBREAK,
            low_to_high_iters=POETIQ_LOW_TO_HIGH_ITERS))
    return voted

def e10_update_task_submission(tid):
    ranked=e10_rank_task(tid)
    E10_FINAL[tid]=poetiq_two_attempts(CHALLENGES[tid],ranked)
    E10_SCHED['voting_updates']+=1
    e10_write_submission()

def e10_write_submission():
    submission=identity_submission(test_challenges)
    if RERUN:
        for tid,rows in E10_FINAL.items():
            if tid in submission and len(submission[tid])==len(rows):submission[tid]=rows
    # In evaluation, preserve the competition-format test fallback.
    write_json_atomic(SUBMISSION_PATH,submission)

E10_METRICS=[]
E10_SCHED_ERROR=None
_E10_KV={'t':-1e9,'v':0.0}
def e10_kv_usage():
    # S1: vLLM KV usage (0..1), read at most every 5 s; 0.0 if the metric is unavailable (gate then stays open)
    now=time.perf_counter()
    if now-_E10_KV['t']>=5.0:
        _E10_KV['t']=now
        try:
            vals=[float(getattr(m,'value',0) or 0) for m in llm.get_metrics() if 'kv_cache_usage' in getattr(m,'name','')]
            _E10_KV['v']=max(vals) if vals else 0.0
        except Exception:
            _E10_KV['v']=0.0
    return _E10_KV['v']

def e10_kv_gate_open():
    if len(E10_ACTIVE)<E10_KV_GATE_MIN_ACTIVE:return True
    kv=e10_kv_usage()
    if kv<=E10_KV_ADMIT_MAX:return True
    E10_SCHED['kv_gate_blocks']+=1;E10_SCHED['kv_gate_last']=round(kv,3)
    return False

def e10_run_solver():
    global E10_TOTAL_GEN
    workers=max(1,POETIQ_SANDBOX_WORKERS)
    with futures.ThreadPoolExecutor(max_workers=workers) as pool:
        last_log=time.perf_counter();drained=False
        while True:
            now=time.perf_counter()
            e10_poll_verifications()
            if now>=HARD_STOP:
                ids=[e10_req(E10_STATES[key],r['phase']) for key,r in E10_ACTIVE.items()]
                if ids:engine.abort_request(ids)
                E10_SCHED['aborted']+=len(ids)
                for req in list(E10_ACTIVE.values()):e10_after_decode(req,'hard_stop',pool)
                break
            if now>=DRAIN_DEADLINE and not drained:
                drained=True
                p1=[r for r in E10_ACTIVE.values() if r['phase']==1]
                if p1:engine.abort_request([e10_req(E10_STATES[r['key']],1) for r in p1])
                for req in p1:
                    req['p1_finish']='drained'
                    if not e10_start_phase2(req,DRAIN_ANSWER_MAX_TOKENS):e10_after_decode(req,'drain_no_room',pool)
            while E10_QUEUE and len(E10_ACTIVE)<CONCURRENCY and now<ADMIT_DEADLINE and not drained and e10_kv_gate_open():
                key=E10_QUEUE[0]
                if E10_STATES[key]['finished']:
                    E10_QUEUE.popleft();continue
                outcome=e10_admit(key)
                if outcome in ('admitted','skipped'):
                    E10_QUEUE.popleft()
                now=time.perf_counter()
            E10_SCHED['max_active']=max(E10_SCHED['max_active'],len(E10_ACTIVE))
            if not E10_ACTIVE:
                if E10_VERIFY:
                    time.sleep(.05);continue
                if not E10_QUEUE or now>=ADMIT_DEADLINE or drained:break
                time.sleep(.01);continue
            outs=engine.step();E10_SCHED['steps']+=1
            for o in outs:
              # S1 (C1): one bad output must not end the whole solver; the unit is closed and the run continues
              try:
                match=re.fullmatch(r'(.+)~e(\d+)~i(\d+)\|([12])',o.request_id)
                if not match:
                    E10_SCHED['unit_errors'].append(f'unexpected request id {o.request_id!r}'[:300]);continue
                tid,ex,iteration,phase=match.group(1),int(match.group(2)),int(match.group(3)),int(match.group(4))
                key=(tid,ex)
                req=E10_ACTIVE.get(key)
                if req is None or req['phase']!=phase:continue
                if E10_STATES[key]['iteration']!=iteration:continue
                comp=o.outputs[0] if o.outputs else None
                if comp is not None:
                    token_ids=list(comp.token_ids);E10_TOTAL_GEN+=len(token_ids)
                    (req['p1_ids'] if phase==1 else req['p2_ids']).extend(token_ids)
                if not o.finished:continue
                reason=comp.finish_reason if comp is not None else None
                if phase==1:
                    req['p1_finish']=reason
                    if THINK_CLOSE_ID in req['p1_ids'] and reason!='length':
                        e10_after_decode(req,'natural',pool)
                    elif not e10_start_phase2(req):
                        e10_after_decode(req,'no_room_for_code',pool)
                else:
                    req['p2_finish']=reason
                    e10_after_decode(req,req['mode'],pool)
              except Exception as _uerr:
                E10_SCHED['unit_errors'].append(f'{getattr(o,"request_id","?")}: {type(_uerr).__name__}: {_uerr}'[:300])
                print('[E10] unit error (isolated):',E10_SCHED['unit_errors'][-1],flush=True)
                try:engine.abort_request([o.request_id])
                except Exception:pass
                _m=re.fullmatch(r'(.+)~e(\d+)~i(\d+)\|([12])',str(getattr(o,'request_id','')))
                if _m:
                    _k=(_m.group(1),int(_m.group(2)))
                    E10_ACTIVE.pop(_k,None)
                    if _k in E10_STATES:
                        E10_STATES[_k]['finished']=True;E10_STATES[_k]['status']='unit_error'
            if time.perf_counter()-last_log >= LOG_EVERY_SEC:
                last_log=time.perf_counter()
                E10_METRICS.append({'t_sec':round(last_log-NOTEBOOK_T0,1),
                                    'active':len(E10_ACTIVE),'queue':len(E10_QUEUE),
                                    'verifying':len(E10_VERIFY),'metrics':metrics_snapshot()})
                print(f"[E10] {(last_log-NOTEBOOK_T0)/60:.1f} min active={len(E10_ACTIVE)} "
                      f"pending={len(E10_QUEUE)} verify={len(E10_VERIFY)} "
                      f"verified_programs={E10_SCHED['train_passed']} tokens={E10_TOTAL_GEN}",flush=True)
        # Harvest verifications that have already finished without launching new iterations.
        # S1 (F2): HARD_STOP is already past at the hard stop, so R5 never harvested verifications started there
        endwait=min(NOTEBOOK_T0+TOTAL_BUDGET_SEC-60,time.perf_counter()+POETIQ_SANDBOX_TIMEOUT*2)
        while E10_VERIFY and time.perf_counter()<endwait:
            e10_poll_verifications()
            if E10_VERIFY:time.sleep(.05)
        E10_SCHED['remaining_verifications']=len(E10_VERIFY)
        # S1 (F2b): leaving the with-block would wait for every queued verification; cancel the queued ones so only
        # the <= POETIQ_SANDBOX_WORKERS running jobs are awaited (GPT audit 2026-10-16)
        pool.shutdown(wait=False,cancel_futures=True)
    E10_SCHED['not_started']=sum(1 for s in E10_STATES.values() if s['iteration']==0 and not s['finished'])
    E10_SCHED['wall_sec']=round(time.perf_counter()-E10_SCHED['started'],1)
    E10_SCHED['tokens']=E10_TOTAL_GEN
    E10_SCHED['tok_per_sec']=round(E10_TOTAL_GEN/max(1,E10_SCHED['wall_sec']),1)

try:
    if ENGINE_READY:
        e10_run_solver()
    else:
        E10_SCHED['skipped']='engine_unavailable'
except BaseException:
    E10_SCHED_ERROR=traceback.format_exc()
    print('[E10] solver error:',E10_SCHED_ERROR[-3500:],flush=True)
    try:
        _ids=[e10_req(E10_STATES[key],r['phase']) for key,r in E10_ACTIVE.items()]
        if _ids:engine.abort_request(_ids)
    except BaseException:pass
finally:
    for _tid in TASK_IDS:
        e10_update_task_submission(_tid)
    E10_SCHED['error']=E10_SCHED_ERROR[-4000:] if E10_SCHED_ERROR else None
    E10_SCHED.setdefault('wall_sec',round(time.perf_counter()-E10_SCHED['started'],1))
    E10_SCHED.setdefault('tokens',E10_TOTAL_GEN)
    try:write_json_atomic(OUT/'poetiq_vllm_metrics.json',E10_METRICS)
    except Exception:pass
    try:write_json_atomic(OUT/'poetiq_final_generated.json',E10_FINAL)
    except Exception:pass
    e10_write_submission()
print('[E10] solver complete',json.dumps({k:v for k,v in E10_SCHED.items() if k!='error'}),flush=True)


In [ ]:
# E10 — Poetiq program synthesis, train-verification and task-level output report
try:
    for tid in TASK_IDS:
        if tid not in E10_FINAL:
            E10_FINAL[tid]=poetiq_two_attempts(CHALLENGES[tid],[])
    generated={tid:E10_FINAL[tid] for tid in TASK_IDS}
    write_json_atomic(OUT/'poetiq_eval_generated_submission.json',generated)
    report={'run_id':RUN_ID,'profile':PROFILE,'rerun':RERUN,
            'source_base_sha256':'530cb119e302899426a2e3f24c5e100279e7167aef77525b6cb93b90a7fc10c0',
            'donor':'https://github.com/poetiq-ai/poetiq-arc-agi-solver',
            'configuration':CONFIG,'engine':ENGINE_INFO,'scheduler':E10_SCHED,
            'tasks_planned':len(TASK_IDS),'queries_planned':sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
            'experts_planned':len(E10_STATES),'experts_train_verified':sum(s['status']=='train_verified' for s in E10_STATES.values()),
            'programs_generated':E10_SCHED['programs_parsed'],
            'expert_states':{f'{t}~e{e}':{'iteration':s['iteration'],'status':s['status'],
                                       'best_score':s['best_score']} for (t,e),s in E10_STATES.items()},
            'task_ranked_programs':{tid:len(e10_rank_task(tid)) for tid in TASK_IDS},
            'generated_attempts_path':'q27x/poetiq_eval_generated_submission.json'}
    if SOLUTIONS is not None:
        hits=0;pass1=0;pass2=0;total=0;records=[]
        for tid in TASK_IDS:
            for qi,att in enumerate(E10_FINAL[tid]):
                gold=SOLUTIONS[tid][qi];a=(att['attempt_1']==gold);b=(att['attempt_2']==gold)
                hits+=a or b;pass1+=a;pass2+=b;total+=1
                records.append({'task':tid,'query':qi,'pass1':bool(a),'pass2':bool(b),'hit':bool(a or b)})
        report.update(eval_query_hits=hits,eval_query_total=total,eval_score_pct=round(100*hits/max(1,total),2),
                      eval_pass1_hits=pass1,eval_pass2_hits=pass2,eval_unit_records=records)
    write_json_atomic(OUT/'poetiq_run_report.json',report)
    print('E10 RUN REPORT:',str(OUT/'poetiq_run_report.json'),
          'eval',report.get('eval_query_hits'),'/ ',report.get('eval_query_total'),flush=True)
    # Last: ensure the actual Kaggle submission and bundled version are identical.
    e10_write_submission()
    _pack=WORKING/'Q27X_E10_POETIQ_BUNDLE.zip'
    with zipfile.ZipFile(_pack,'w',zipfile.ZIP_DEFLATED) as z:
        for fp in (SUBMISSION_PATH, OUT/'poetiq_run_report.json',OUT/'poetiq_final_generated.json',
                   OUT/'poetiq_eval_generated_submission.json',OUT/'poetiq_iterations.jsonl',
                   OUT/'poetiq_vllm_metrics.json'):
            if fp.exists():z.write(fp,arcname=fp.name)
    print('E10 bundle:',_pack,flush=True)
except BaseException:
    print('E10 reporting error (submission.json still exists):',traceback.format_exc()[-3000:],flush=True)


In [ ]:
# ==================================================================== CELL 8b — E18: engine stats, stop the vLLM server
try:
    _st = {"engine": ENGINE_INFO.get("engine"), "shim": getattr(engine, "stats", None), "concurrency": CONCURRENCY,
           "kv_tokens": ENGINE_INFO.get("kv_tokens"), "scheduler_tok_per_sec": E10_SCHED.get("tok_per_sec"),
           "scheduler_tokens": E10_SCHED.get("tokens"), "max_active": E10_SCHED.get("max_active"),
           "kv_gate_blocks": E10_SCHED.get("kv_gate_blocks"), "metrics_end": metrics_snapshot() if ENGINE_READY else []}
    write_json_atomic(OUT / "e18_engine_stats.json", _st)
    print("E18 engine stats:", json.dumps({k: v for k, v in _st.items() if k != "metrics_end"}, default=str)[:2000], flush=True)
    # rev 2 (audit B-H1): the KV pressure at 20 streams
    _keys = ("num_preemptions", "num_requests_waiting", "kv_cache_usage", "prefix_cache_hits", "prefix_cache_queries")
    print("E18 KV / preemption:", json.dumps({m["name"] + (m.get("labels") or ""): m.get("value") for m in _st["metrics_end"]
                                               if any(k in str(m.get("name")) for k in _keys)})[:1500], flush=True)
except BaseException:
    print("E18 stats error:", traceback.format_exc()[-1500:], flush=True)
try:
    if globals().get("E18_SERVER") is not None:
        E18_SERVER.kill(fast=RERUN)          # rev 2 (B-L4): at once after the rerun's hard stop
        print("E18: vLLM 0.32 server stopped", flush=True)
except BaseException as _e:
    print("E18: server stop error", repr(_e)[:300], flush=True)
